<a href="https://colab.research.google.com/github/Tr3vour-svg/MEDIC-ASTRO/blob/main/AGENTIC_MEDICAL_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dependencies


In [ ]:
!pip install -U --quiet langchain-text-splitters
!pip install -U --quiet langchain-tavily
!pip install -U --quiet turso_serverless
!pip install -U --quiet \
    langgraph \
    langchain \
    langchain-core \
    langchain-openai \
    langchain-community \
    pandas

API keys

In [ ]:
import os
from google.colab import userdata

for key in ("TURSO_DATABASE_URL", "TURSO_AUTH_TOKEN"):
    if key not in os.environ:
        os.environ[key] = userdata.get(key)

if "TURSO_PATIENTS_URL" not in os.environ:
    os.environ["TURSO_PATIENTS_URL"] = userdata.get("TURSO_PATIENTS_URL")

if "TURSO_PATIENTS_TOKEN" not in os.environ:
    os.environ["TURSO_PATIENTS_TOKEN"] = userdata.get("TURSO_PATIENTS_TOKEN")


print("✅ Turso credentials loaded.")

# Secure API Key Configuration from Colab Secrets
def _load_secret(name, env_var=None):
    """Load a secret from Colab Secrets into an environment variable."""
    env_var = env_var or name
    if env_var not in os.environ:
        try:
            os.environ[env_var] = userdata.get(name)
        except Exception as e:
            print(f"⚠️  Could not load secret '{name}': {e}")
            return False
    return True

# Load all required API keys
secrets = {
    "OPENAI_API_KEY": "OPENAI_API_KEY",
    "PINECONE_API_KEY": "PINECONE_API_KEY",
    "LANGSMITH_API_KEY": "LANGSMITH_API_KEY",
    "DEEPSEEK_API_KEY": "DEEPSEEK_API_KEY",
    "OPENROUTER_API_KEY": "OPENROUTER_API_KEY",
    "TAVILY_API_KEY" : "TAVILY_API_KEY",
}

loaded = [name for name in secrets if _load_secret(name)]

# Enable LangSmith tracing if the key is present
if "LANGSMITH_API_KEY" in os.environ:
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ.setdefault("LANGSMITH_PROJECT", "default")

print(f"✅ Packages installed and API keys configured successfully!")
print(f"   Loaded: {', '.join(loaded) if loaded else 'none'}")

Configuration & Environment Setup

In [ ]:

import os
from google.colab import drive

# 1. Mount Google Drive (if not already mounted)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Define Dataset Paths (Adjust folder name if saved elsewhere in your Drive)
DATA_DIR = '/content/drive/MyDrive/Hospital_Mock_Data_2GB'  # Update if your folder has a different name

PATHS = {
    "staff_csv":    os.path.join(DATA_DIR, "staff_directory.csv"),
    "salary_json":  os.path.join(DATA_DIR, "salary_structure.json"),
    "policy_json":  os.path.join(DATA_DIR, "hospital_policy_chunks.json"),
    "patient_json": os.path.join(DATA_DIR, "patient_records.json"),
}

# Verify files exist
print("📁 Verifying dataset files in Google Drive:")
for key, path in PATHS.items():
    exists = os.path.exists(path)
    status = "✅ Found" if exists else "❌ Missing (Check path)"
    print(f" - {key}: {status} ({path})")

# 3. Global Constants for Models and Vector DB
CONFIG = {
    "llm_model": "gpt-4o",                       # Frontier model for orchestrator/reasoning
    "cheap_llm_model": "gpt-4o-mini",            # Small model for routing/simple queries
    "reviewer_model": "deepseek-chat",           # DeepSeek reviewer model
    "embedding_model": "text-embedding-3-small", # OpenAI embedding model for policies
    "pinecone_index_name": "hospital-policies",  # Pinecone index name
    "pinecone_dimension": 1536,                  # Dimension size for text-embedding-3-small
}

# 4. Shared utilities (used by every node cell below)
def _message_text(message) -> str:
    """Extract text from a message object/dict; tolerates list-of-blocks content."""
    content = (
        message.get("content") if isinstance(message, dict)
        else getattr(message, "content", "")
    )
    if isinstance(content, list):
        content = " ".join(
            b.get("text", "") if isinstance(b, dict) else str(b) for b in content
        )
    return content if isinstance(content, str) else str(content or "")


print("\n⚙️ Global Configuration Loaded Successfully!")
print("🔧 Shared helpers defined: _message_text")

Data Audit & Cleaning Script

In [ ]:

import pandas as pd
import json
import os

print("🔍 Starting Data Audit & Cleanliness Check...\n")

# ----------------------------------------------------------------------
# 1. Audit staff_directory.csv (~100 MB)
# ----------------------------------------------------------------------
print("="*50)
print("1. AUDITING: staff_directory.csv")
print("="*50)
staff_path = PATHS["staff_csv"]

if os.path.exists(staff_path):
    total_staff_rows = 0
    null_counts = None
    duplicate_ids = 0

    # Stream in chunks to protect RAM
    for chunk in pd.read_csv(staff_path, chunksize=100000):
        total_staff_rows += len(chunk)
        if null_counts is None:
            null_counts = chunk.isnull().sum()
        else:
            null_counts += chunk.isnull().sum()
        duplicate_ids += chunk['emp_id'].duplicated().sum()

    print(f"Total Rows: {total_staff_rows}")
    print(f"Duplicate Employee IDs Found: {duplicate_ids}")
    print("Missing Values per Column:")
    print(null_counts)
else:
    print("❌ staff_directory.csv not found at path.")


# ----------------------------------------------------------------------
# 2. Audit salary_structure.json (~80 MB)
# ----------------------------------------------------------------------
print("\n" + "="*50)
print("2. AUDITING: salary_structure.json")
print("="*50)
salary_path = PATHS["salary_json"]

if os.path.exists(salary_path):
    file_size_mb = os.path.getsize(salary_path) / (1024 * 1024)
    print(f"File Size: {file_size_mb:.2f} MB")

    # Quick structural check by loading first few items or validating JSON formatting
    try:
        with open(salary_path, 'r') as f:
            sample_data = json.load(f) if file_size_mb < 200 else "Large JSON (Stream validated)"
        print(f"✅ Salary JSON structure is valid.")
    except Exception as e:
        print(f"❌ JSON Parsing Error in salary structure: {e}")
else:
    print("❌ salary_structure.json not found.")


# ----------------------------------------------------------------------
# 3. Audit hospital_policy_chunks.json (~300 MB)
# ----------------------------------------------------------------------
print("\n" + "="*50)
print("3. AUDITING: hospital_policy_chunks.json")
print("="*50)
policy_path = PATHS["policy_json"]

if os.path.exists(policy_path):
    policy_size_mb = os.path.getsize(policy_path) / (1024 * 1024)
    print(f"File Size: {policy_size_mb:.2f} MB")

    # Check sample keys for RAG suitability
    with open(policy_path, 'r') as f:
        # Load a small snippet safely or inspect schema
        sample_policies = json.load(f)
        sample_chunk = sample_policies[0] if isinstance(sample_policies, list) else None

    if sample_chunk and all(k in sample_chunk for k in ["policy_id", "category", "content"]):
        print(f"✅ Policy schema valid. Total Chunks: {len(sample_policies)}")
    else:
        print("⚠️ Policy schema missing expected RAG keys (policy_id, category, content).")
else:
    print("❌ hospital_policy_chunks.json not found.")


# ----------------------------------------------------------------------
# 4. Audit patient_records.json (~1.5 GB)
# ----------------------------------------------------------------------
print("\n" + "="*50)
print("4. AUDITING: patient_records.json")
print("="*50)
patient_path = PATHS["patient_json"]

if os.path.exists(patient_path):
    patient_size_gb = os.path.getsize(patient_path) / (1024 * 1024 * 1024)
    print(f"File Size: {patient_size_gb:.2f} GB")
    print("✅ Patient records file detected and ready for indexing.")
else:
    print("❌ patient_records.json not found.")

print("\n" + "="*50)
print("🎯 DATA AUDIT COMPLETE")
print("="*50)

 Turso Connection Verification

In [ ]:
# Cell 5b: Turso Connection Verification (replaces DuckDB)
import turso_serverless

print("🌐 Verifying Turso Cloud connection...")

def _turso_connect():
    """Open a short-lived remote connection to Turso Cloud."""
    return turso_serverless.connect(
        os.environ["TURSO_DATABASE_URL"],
        auth_token=os.environ["TURSO_AUTH_TOKEN"],
    )

# Smoke-test: list tables and row counts
conn = _turso_connect()
try:
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
    ).fetchall()
    print(f"📊 Tables found: {[t[0] for t in tables]}")

    for (table_name,) in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        print(f" - {table_name}: {count:,} rows")
finally:
    conn.close()

print("✅ Turso Cloud is ready.")

Advanced Semantic Chunking Pipeline

In [ ]:
import json
import os
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

# ----------------------------------------------------------------------
# Cache check: reuse chunks if already saved to Drive
# ----------------------------------------------------------------------
chunked_output_path = os.path.join(DATA_DIR, "hospital_policy_competent_chunks.json")

if os.path.exists(chunked_output_path):
    print(f"♻️ Found cached chunks at {chunked_output_path}, loading...")
    with open(chunked_output_path, "r") as f:
        chunked_documents = json.load(f)
    print(f"✅ Loaded {len(chunked_documents):,} chunks from cache. Skipping re-chunking.")
else:
    print("📄 No cache found. Loading raw policy documents...")

    policy_path = PATHS["policy_json"]
    if not os.path.exists(policy_path):
        raise FileNotFoundError(f"Policy file not found at {policy_path}")

    with open(policy_path, "r") as f:
        raw_policies = json.load(f)

    print(f"Loaded {len(raw_policies):,} raw policy documents.")

    # ------------------------------------------------------------------
    # Stage 1: split by Markdown headers (preserves section context)
    # ------------------------------------------------------------------
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False,  # keep headers in chunk text for context
    )

    # ------------------------------------------------------------------
    # Stage 2: further split oversized sections
    # ------------------------------------------------------------------
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunked_documents = []

    print("⚙️ Processing and chunking documents with header awareness...")

    for policy in raw_policies:
        content = policy.get("content", "")
        policy_id = policy.get("policy_id", "POL-UNKNOWN")
        category = policy.get("category", "General")
        department = policy.get("target_department", "All")
        title = policy.get("title", "Untitled Policy")

        # Stage 1: header-based split
        header_sections = header_splitter.split_text(content)

        chunk_index = 0
        for section in header_sections:
            section_text = section.page_content
            section_meta = section.metadata  # e.g. {"h1": "...", "h2": "..."}

            # Stage 2: split sections that exceed chunk_size
            if len(section_text) > 800:
                splits = text_splitter.split_text(section_text)
            else:
                splits = [section_text]

            for split_text in splits:
                chunk_index += 1
                chunk_record = {
                    "chunk_id": f"{policy_id}-chk{chunk_index}",
                    "parent_policy_id": policy_id,
                    "title": title,
                    "category": category,
                    "department": department,
                    # Preserve header hierarchy as metadata
                    "header_path": " > ".join(
                        v for v in [section_meta.get("h1"),
                                    section_meta.get("h2"),
                                    section_meta.get("h3")] if v
                    ),
                    "content": split_text,
                    "char_length": len(split_text),
                }
                chunked_documents.append(chunk_record)

    print(f"\n✅ Chunking Complete!")
    print(f" - Original Documents: {len(raw_policies):,}")
    print(f" - Final Semantic Chunks Generated: {len(chunked_documents):,}")

    # ------------------------------------------------------------------
    # Persist chunks to Google Drive for reuse
    # ------------------------------------------------------------------
    with open(chunked_output_path, "w") as f:
        json.dump(chunked_documents, f, indent=2)

    print(f"\n💾 Saved chunked data to Google Drive: {chunked_output_path}")

# ----------------------------------------------------------------------
# Inspect sample chunks
# ----------------------------------------------------------------------
print("\n🔍 Sample Chunk Inspection:")
for sample in chunked_documents[:2]:
    print("-" * 40)
    print(f"Chunk ID: {sample['chunk_id']} | Header Path: {sample['header_path']} | Length: {sample['char_length']} chars")
    print(f"Content Preview:\n{sample['content'][:300]}...")

Initialize Pinecone & Embedding Model

In [ ]:
!pip install -U --quiet langchain-pinecone

In [ ]:

import os
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone.rerank import PineconeRerank
from langchain_openai import OpenAIEmbeddings

print("🌲 Initializing Pinecone Client & Embedding Model...")

# 1. Initialize Pinecone Client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = CONFIG["pinecone_index_name"]
dimension = CONFIG["pinecone_dimension"]

# 2. Ensure Pinecone index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    print(f"Creating new Pinecone serverless index: '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("✅ Index created successfully.")
else:
    print(f"⚡ Pinecone index '{index_name}' already exists.")

# 3. Initialize Embeddings model via OpenRouter
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",          # OpenRouter model name format
    api_key=os.environ["OPENROUTER_API_KEY"],        # Use OpenRouter key
    base_url="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,                # Required for non-OpenAI endpoints
)
print("✅ Embeddings model ('text-embedding-3-small') loaded successfully!")

Convert Chunks to LangChain Documents & Upsert to Pinecone

In [ ]:

import json
import os
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore

print("🔄 Loading competent chunks from Google Drive...")

chunked_output_path = os.path.join(DATA_DIR, "hospital_policy_competent_chunks.json")
if not os.path.exists(chunked_output_path):
    raise FileNotFoundError(f"Chunked file not found at {chunked_output_path}. Run Cell 5 first!")

with open(chunked_output_path, 'r') as f:
    chunked_data = json.load(f)

print(f"Loaded {len(chunked_data):,} chunks. Converting to LangChain Documents...")

# 1. Convert dictionary chunks into LangChain Document objects
#    - Read content without popping (safe on rerun)
#    - Use chunk_id as Pinecone vector ID (idempotent upsert)
langchain_docs = []
doc_ids = []
for item in chunked_data:
    metadata = {
        "chunk_id": item.get("chunk_id"),
        "parent_policy_id": item.get("parent_policy_id"),
        "title": item.get("title"),
        "category": item.get("category"),
        "department": item.get("department"),
        "header_path": item.get("header_path", ""),
        "char_length": item.get("char_length"),
    }
    langchain_docs.append(Document(page_content=item.get("content", ""), metadata=metadata))
    doc_ids.append(item.get("chunk_id"))

print(f"✅ Converted {len(langchain_docs):,} items into LangChain Document objects.")

# 2. Connect to Pinecone Vector Store & Check Status
vector_store = PineconeVectorStore(index_name=CONFIG["pinecone_index_name"], embedding=embeddings)

index_stats = pc.Index(CONFIG["pinecone_index_name"]).describe_index_stats()
total_vectors = index_stats.get('total_vector_count', 0)
expected_count = len(langchain_docs)

# 3. Idempotent upsert: run if index is empty OR count mismatch (partial upload)
if total_vectors < expected_count:
    if total_vectors == 0:
        print(f"🚀 Upserting {expected_count:,} documents into Pinecone in batches...")
    else:
        print(f"⚠️  Index has {total_vectors:,} vectors but local has {expected_count:,}. "
              f"Running idempotent upsert to reconcile...")

    # Pinecone upsert limit: 1000 records or 2MB per request → 400 is a safe margin
    batch_size = 400
    for i in range(0, len(langchain_docs), batch_size):
        batch_docs = langchain_docs[i:i + batch_size]
        batch_ids = doc_ids[i:i + batch_size]
        vector_store.add_documents(batch_docs, ids=batch_ids)
        print(f"  - Progress: {min(i + batch_size, expected_count)}/{expected_count} documents upserted.")

    print("🎉 All documents successfully vectorized and indexed in Pinecone!")
else:
    print(f"⚡ Pinecone index already contains {total_vectors:,} vectors (>= {expected_count:,} local). "
          f"Skipping upload (cached).")

Define the LangGraph State Schema

In [ ]:

from typing import TypedDict, Annotated, Optional, List
from operator import add
from langgraph.graph import add_messages

print("📦 Defining LangGraph HospitalAgentState schema...")

# --- Custom reducers -------------------------------------------------------
# Each agent appends its own contribution instead of clobbering prior data.

def merge_contexts(left, right):
    """
    Append-only within a turn; a '__reset__' marker clears history.
    """
    if right and any(
        isinstance(f, dict) and f.get("source") == "__reset__"
        for f in right
    ):
        return [f for f in right if f.get("source") != "__reset__"]
    return (left or []) + (right or [])

def merge_flags(left: Optional[dict], right: Optional[dict]) -> dict:
    """Merge per-agent status flags without overwriting unrelated keys."""
    return {**(left or {}), **(right or {})}

def keep_last_non_none(left, right):
    """For scalar fields: keep latest non-None value (routing fields are single-writer)."""
    return right if right is not None else left


class HospitalAgentState(TypedDict):
    # Chat history — append-only via add_messages reducer
    messages: Annotated[list, add_messages]

    # User Security & Authorization Context (set once at entry, never mutated by agents)
    user_id: str
    user_clearance: str
    user_department: Optional[str]

    # Workflow Orchestration — single-writer fields (supervisor only)
    intent: Annotated[Optional[str], keep_last_non_none]
    is_safe: Annotated[bool, keep_last_non_none]
    next_agent: Annotated[Optional[str], keep_last_non_none]

    # Multi-writer fields — every agent APPENDS, none overwrites
    # Each entry: {"source": "<agent_name>", "content": "<text>", "metadata": {...}}
    retrieved_context: Annotated[List[dict], merge_contexts]

    # Per-agent status / diagnostics (e.g. {"policy_agent": "ok", "hr_agent": "no_data"})
    agent_status: Annotated[dict, merge_flags]

    # Final sanitized response (writer: output_guardrail only)
    final_output: Annotated[Optional[str], keep_last_non_none]

    revision_count: Annotated[int, keep_last_non_none]   # or just int with default 0


print("✅ HospitalAgentState defined successfully!")

Define MCP-Style Tool Functions (Policies, Staff, Salaries, Patients)

In [ ]:
# Cell 9: Define Secure MCP-Style Tool Functions (Turso + Tavily)
!pip install -U --quiet langchain-community langchain-tavily turso_serverless

import os
import json
import turso_serverless
from langchain_core.tools import tool
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_tavily import TavilySearch

print("🛠️ Setting up MCP-style tool functions (Turso + Tavily)...")

# ─────────────────────────────────────────────────────────────────────────────
# Shared embeddings / vector store
# ─────────────────────────────────────────────────────────────────────────────
if "embeddings" not in globals():
    embeddings = OpenAIEmbeddings(
        model="openai/text-embedding-3-small",
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
        check_embedding_ctx_length=False,
        tiktoken_model_name="text-embedding-3-small",
    )

if "vector_store" not in globals():
    vector_store = PineconeVectorStore(
        index_name=CONFIG["pinecone_index_name"],
        embedding=embeddings,
    )

# ─────────────────────────────────────────────────────────────────────────────
# Turso connection helper (short-lived per query)
# ─────────────────────────────────────────────────────────────────────────────
def _turso():
    """Open a fresh remote connection to Turso Cloud."""
    return turso_serverless.connect(
        os.environ["TURSO_DATABASE_URL"],
        auth_token=os.environ["TURSO_AUTH_TOKEN"],
    )


# ----------------------------------------------------------------------
# Policy Search (Pinecone vector store)
# ----------------------------------------------------------------------
@tool
def search_hospital_policies(query: str, department: str = None) -> str:
    """Search hospital operational policies, HR guidelines, and clinical handbooks
    using semantic vector search. Optionally filter by department (e.g. 'Nursing', 'HR')."""
    filter_expr = None
    if department:
        filter_expr = {"department": {"$in": ["All", department]}}

    docs = vector_store.similarity_search(query, k=3, filter=filter_expr)
    if not docs:
        return "No relevant hospital policies found."

    results = []
    for i, doc in enumerate(docs):
        title = doc.metadata.get("title", "Policy")
        dept = doc.metadata.get("department", "All")
        header = doc.metadata.get("header_path", "")
        header_str = f" | Section: {header}" if header else ""
        results.append(
            f"Source {i+1} [{title} - Dept: {dept}{header_str}]:\n{doc.page_content}"
        )
    return "\n\n".join(results)


# ----------------------------------------------------------------------
# Staff Directory (Turso / SQLite-compatible SQL)
# ----------------------------------------------------------------------
@tool
def query_staff_directory(department: str = None, role: str = None) -> str:
    """Query the staff directory to find employee details, department mappings,
    and reporting hierarchies."""
    clauses, params = [], []

    if department:
        clauses.append("department LIKE ?")
        params.append(f"%{department}%")
    elif role:
        clauses.append("role LIKE ?")
        params.append(f"%{role}%")

    where = (" WHERE " + " AND ".join(clauses)) if clauses else ""
    sql = (
        "SELECT emp_id, name, role, department "
        f"FROM staff{where} LIMIT 10"
    )

    conn = _turso()
    try:
        cur = conn.cursor()
        cur.execute(sql, tuple(params))
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    if not rows:
        return "No matching staff records found."

    header = "| " + " | ".join(cols) + " |"
    sep = "|" + "|".join(["---"] * len(cols)) + "|"
    body = "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in rows)
    return "\n".join([header, sep, body])


# ----------------------------------------------------------------------
# Salary Structure (Turso / JSON field parsing)
# ----------------------------------------------------------------------
@tool
def query_salary_structure(emp_id: str) -> str:
    """Query employee compensation for a given emp_id.
    The caller (graph node) is responsible for verifying user clearance
    before invoking this tool."""
    conn = _turso()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM salaries WHERE emp_id = ?", (emp_id,))
        rows = cur.fetchall()
        if not rows:
            return f"No salary record found for Employee ID {emp_id}."
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    record = dict(zip(cols, rows[0]))

    allowances = record.get("allowances")
    if isinstance(allowances, str):
        try:
            allowances = json.loads(allowances)
        except Exception:
            allowances = {"raw": allowances}

    allowance_str = ", ".join(f"{k}: {v}" for k, v in (allowances or {}).items())
    base = record.get("base_salary_usd", 0) or 0
    return (
        f"Salary details for {emp_id}:\n"
        f"- Tier: {record.get('tier')}\n"
        f"- Base Salary: ${base:,}\n"
        f"- Bonus Eligible: {record.get('bonus_eligible')}\n"
        f"- Allowances: {allowance_str}"
    )


# ----------------------------------------------------------------------
# Patient Records (Turso, with partial PHI masking)
# ----------------------------------------------------------------------
@tool
def query_patient_records(patient_id: str) -> str:
    """Query patient admission, room, and billing status.
    Name and DOB are unmasked for identification. Clinical notes are masked."""
    conn = _turso()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM patients WHERE patient_id = ?", (patient_id,))
        rows = cur.fetchall()
        if not rows:
            return f"No patient found with ID {patient_id}."
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    record = dict(zip(cols, rows[0]))

    # Mask ONLY the highly sensitive clinical free-text fields.
    # Name and DOB are left visible for medical personnel to identify the patient.
    for phi in ("diagnosis", "treatment_notes"):
        if phi in record:
            record[phi] = "[REDACTED_PHI]"

    charges = record.get("total_charges_usd", 0) or 0
    return (
        f"Patient details for {patient_id}:\n"
        f"- Name: {record.get('full_name')}\n"
        f"- DOB: {record.get('dob')}\n"
        f"- Admission Date: {record.get('admission_date')}\n"
        f"- Room: {record.get('room_number')}\n"
        f"- Primary Physician: {record.get('primary_physician_id')}\n"
        f"- Diagnosis: {record.get('diagnosis')}\n"
        f"- Treatment Notes: {record.get('treatment_notes')}\n"
        f"- Billing Status: {record.get('billing_status')}\n"
        f"- Total Charges: ${charges:,}"
    )


# ----------------------------------------------------------------------
# External Web Search (Tavily)
# ----------------------------------------------------------------------
_tavily = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    api_key=os.environ["TAVILY_API_KEY"]
)

@tool
def external_web_search(query: str) -> str:
    """Search the open web for external medical literature, clinical definitions,
    and international referral hospital directories."""
    try:
        response = _tavily.invoke({"query": query})
        if not response.get("results"):
            return "No relevant web results found."

        formatted = []
        for i, res in enumerate(response["results"], 1):
            title = res.get("title", "Source")
            content = res.get("content", "")
            url = res.get("url", "")
            formatted.append(f"Source {i} [{title} - {url}]:\n{content}")

        return "\n\n".join(formatted)
    except Exception as e:
        return f"Web search failed: {e}"


# Bundle all tools for easy agent binding
ALL_TOOLS = [
    search_hospital_policies,
    query_staff_directory,
    query_salary_structure,
    query_patient_records,
    external_web_search,
]

print(f"✅ Successfully registered {len(ALL_TOOLS)} secure MCP-style tools (Turso + Tavily).")

Initialize ChatOpenAI and Bind All Tools

In [ ]:

from langchain_openai import ChatOpenAI

print("🤖 Initializing GPT-4o via OpenRouter and binding tools...")

# Initialize the orchestrator model through OpenRouter
llm = ChatOpenAI(
    model="openai/gpt-4o",                        # OpenRouter model slug format
    api_key=os.environ["OPENROUTER_API_KEY"],     # Loaded from Colab Secrets
    base_url="https://openrouter.ai/api/v1",      # OpenRouter OpenAI-compatible endpoint
    temperature=0,
)

# Bind all MCP tools — same bind_tools API as native OpenAI
llm_with_tools = llm.bind_tools(ALL_TOOLS)

print(f"✅ LLM initialized via OpenRouter and all {len(ALL_TOOLS)} MCP tools bound successfully!")

Define Input Guardrail & Security Authorization Node

In [ ]:

print("🛡️ Defining Input Guardrail & Security Authorization Node...")

# Canonical clearance levels — anything outside this set is rejected.
VALID_CLEARANCES = {"Executive", "Department Head", "Physician", "Intern"}

# Common prompt injection / jailbreak markers (first line of defense, not the only one)
JAILBREAK_KEYWORDS = [
    "ignore previous instructions",
    "disregard prior directions",
    "system update:",
    "bypass security",
    "override rbac",
    "you are now",
    "act as if you are",
]


def input_guardrail_node(state: HospitalAgentState) -> dict:
    """
    1. Validate identity context (user_id, user_clearance) — zero-trust entry check.
    2. Scan user input for prompt injection / jailbreak patterns.
    3. Set is_safe so downstream nodes can branch on it.
    Returns ONLY the fields it owns; never touches retrieved_context.
    """
    messages = state.get("messages") or []
    latest_message = messages[-1].content if messages else ""
    user_id = state.get("user_id", "")
    user_clearance = state.get("user_clearance", "")

    # ---- (1) Identity / clearance validation -------------------------------
    if not user_id:
        return {
            "is_safe": False,
            "final_output": "SECURITY WARNING: Missing user identity. Request blocked.",
            "intent": "blocked",
            "agent_status": {"input_guardrail": "no_identity"},
        }

    if user_clearance not in VALID_CLEARANCES:
        return {
            "is_safe": False,
            "final_output": (
                f"SECURITY WARNING: Unknown clearance level '{user_clearance}'. "
                f"Request blocked."
            ),
            "intent": "blocked",
            "agent_status": {"input_guardrail": "invalid_clearance"},
        }

    # ---- (2) Prompt injection scan -----------------------------------------
    lowercased_input = latest_message.lower()
    detected = next(
        (kw for kw in JAILBREAK_KEYWORDS if kw in lowercased_input),
        None,
    )

    if detected:
        print(f"🚨 SECURITY ALERT: Prompt injection pattern detected ('{detected}')")
        return {
            "is_safe": False,
            "final_output": (
                "SECURITY WARNING: Your input triggered our automated prompt "
                "injection guardrail. Request blocked."
            ),
            "intent": "blocked",
            "agent_status": {"input_guardrail": f"injection:{detected}"},
        }

    # ---- (3) Safe path -----------------------------------------------------
    print("✅ Input passed security guardrail verification.")
    return {
        "is_safe": True,
        # Do NOT write intent=None — supervisor is the single writer for intent.
        # Leaving it out preserves whatever the supervisor will set next.
        "agent_status": {"input_guardrail": "ok"},
    }


print("✅ Input guardrail node defined successfully!")

Define MEDIC ASTRO — Supervisor & Orchestrator Node

In [ ]:
# Cell 14: MEDIC ASTRO (Supervisor / Orchestrator Node)
import os
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

print("👔 Defining MEDIC ASTRO (Supervisor / Orchestrator Node)...")

# Fast routing model via OpenRouter
supervisor_llm = ChatOpenAI(
    model=CONFIG["cheap_llm_model"],              # e.g. "openai/gpt-4o-mini"
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

VALID_INTENTS = {"policy", "hr_salary", "staff", "patient", "web_search", "general"}

# Improved prompt with concrete examples to prevent "general" fallback
MEDIC_ASTRO_PROMPT = (
    "You are MEDIC ASTRO, the Chief Medical Supervisor Agent for a secure hospital. "
    "Your job is to route the user's LATEST message to exactly one specialist agent.\n\n"
    "Categories:\n"
    "1. 'policy'     -> Hospital guidelines, rules, SOPs, compliance, code protocols, handbooks.\n"
    "                   e.g. 'What is the leave policy?', 'What is the escalation SOP?'\n"
    "2. 'hr_salary'  -> Salary, compensation, bonuses, allowances for a specific employee.\n"
    "                   e.g. 'What is the salary for EMP-000001?'\n"
    "3. 'staff'      -> Staff directory, who works where, roles, reporting lines.\n"
    "                   e.g. 'Who is the head of Cardiology?', 'Find EMP-000009'.\n"
    "4. 'patient'    -> Patient records, admissions, medical notes, billing status, AND the physician assigned to the patient."
    "                   e.g. 'Where is PAT-0000001 admitted?'\n"
    "5. 'web_search' -> External medical literature, definitions, or referral hospitals NOT in our internal system.\n"
    "                   e.g. 'What are the latest WHO sepsis guidelines?'\n"
    "6. 'general'    -> Greetings, chit-chat, or queries that fit NONE of the above.\n\n"
    "RULES:\n"
    "- When in doubt between 'policy' and 'general', ALWAYS prefer 'policy'. Hospital questions are usually operational.\n"
    "- The caller's clearance is provided. Use it ONLY to decide if a sensitive intent (like 'hr_salary') is appropriate.\n\n"
    "Respond with ONLY one exact keyword: policy, hr_salary, staff, patient, web_search, general."
)


def _parse_intent(raw: str) -> str:
    """Extract the first valid intent token from a possibly noisy LLM response."""
    if not raw:
        return ""
    # Lowercase, strip punctuation, look for any valid intent as a word
    cleaned = raw.strip().lower().strip(".,!?\"'`").split()
    for token in cleaned:
        if token in VALID_INTENTS:
            return token
    return ""


def medic_astro_node(state: HospitalAgentState) -> dict:
    """
    MEDIC ASTRO — inspects the latest user message plus clearance context and
    classifies the intent. Single writer of the `intent` field (per Cell 8 reducer).
    """
    messages = state.get("messages") or []
    if not messages:
        return {
            "intent": "general",
            "agent_status": {"medic_astro": "empty_input"},
        }

    user_clearance = state.get("user_clearance", "Unknown")

    # Construct the prompt array: System rules -> Clearance context -> Latest User Message
    supervisor_messages = [
        SystemMessage(content=MEDIC_ASTRO_PROMPT),
        SystemMessage(content=f"Caller clearance level: {user_clearance}"),
        messages[-1]  # ONLY pass the latest message, not the whole history
    ]

    try:
        response = supervisor_llm.invoke(supervisor_messages)
        detected_intent = _parse_intent(response.content)
    except Exception as e:
        print(f"⚠️ MEDIC ASTRO routing error: {e}")
        detected_intent = ""
        response = None

    if detected_intent not in VALID_INTENTS:
        # Do NOT silently default to "policy" — surface the failure instead
        raw_content = response.content if response else "NO_RESPONSE"
        print(f"⚠️ MEDIC ASTRO could not parse intent (raw='{raw_content!r}'). "
              f"Defaulting to 'general' for safety.")
        detected_intent = "general"

    print(f"🧭 MEDIC ASTRO routed intent -> [{detected_intent.upper()}]")

    return {
        "intent": detected_intent,
        "agent_status": {"medic_astro": f"routed:{detected_intent}"},
    }


print("✅ MEDIC ASTRO (Supervisor Node) defined successfully!")

Policy Retriever + Inline Indirect-Injection Sanitizer

In [ ]:
# Cell 14: Policy Retriever + Reranker + Inline Sanitizer (Hardened)
import re
import unicodedata
from langchain_pinecone.rerank import PineconeRerank

print("🔎 Defining Policy Retriever with Reranker + Inline Sanitizer...")

# ---- Reranker initialization -------------------------------------------------
reranker = PineconeRerank(
    model="bge-reranker-v2-m3",
    top_n=3,
)

# ---- Tunables ----------------------------------------------------------------
INITIAL_CANDIDATE_K = 15     # bumped up for better recall before rerank
MIN_RERANK_SCORE = 0.001     # bge-reranker-v2-m3 returns small scores (~0.01)
MIN_RELEVANCE_SCORE = 0.10   # used only on the vector-score fallback path

# Departments that actually exist in the dataset + "All"
KNOWN_DEPARTMENTS = {
    "Administration", "Cardiology", "Neurology", "Pediatrics",
    "Oncology", "Radiology", "Emergency", "ICU", "HR",
    "Nursing", "Surgery", "Pharmacy",
    "All",
}

# ---- Indirect-injection patterns --------------------------------------------
INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?(previous|prior)\s+instructions?",
    r"disregard\s+(all\s+)?(prior|previous)\s+(rules|directions?)",
    r"system\s*(prompt|update|override)\s*:",
    r"override\s+(permissions?|rbac|security)",
    r"you\s+are\s+now\s+(a|an|the)\s+",
]


# ----------------------------------------------------------------------
# Sanitization helpers
# ----------------------------------------------------------------------
def _normalize_for_scan(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]", "", text)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    return text.lower()


def _scan_for_injection(text: str) -> str | None:
    normalized = _normalize_for_scan(text)
    for pattern in INJECTION_PATTERNS:
        if re.search(pattern, normalized):
            return pattern
    return None


# ----------------------------------------------------------------------
# Query builder
# ----------------------------------------------------------------------
def _build_retrieval_query(state: HospitalAgentState) -> str:
    messages = state.get("messages") or []
    if not messages:
        return ""
    raw = _message_text(messages[-1])
    return re.sub(r"\s+", " ", raw).strip().rstrip("?.! ")


# ----------------------------------------------------------------------
# Department resolver — skips filter if UI value isn't in the index
# ----------------------------------------------------------------------
def _resolve_department(user_department: str | None) -> str | None:
    if not user_department:
        return None
    if user_department in KNOWN_DEPARTMENTS:
        return user_department
    for d in KNOWN_DEPARTMENTS:
        if d.lower() == user_department.lower():
            return d
    print(f"⚠️ '{user_department}' is not a known department — skipping filter.")
    return None


# ----------------------------------------------------------------------
# Fallback retriever (used only if reranker fails)
# ----------------------------------------------------------------------
def _retrieve_with_scores(query: str, k: int, department: str | None) -> list:
    filter_expr = {"department": {"$in": ["All", department]}} if department else None
    try:
        return vector_store.similarity_search_with_relevance_scores(
            query, k=k, filter=filter_expr
        )
    except Exception as e:
        print(f"⚠️ Scored retrieval failed, falling back to plain search: {e}")
        try:
            docs = vector_store.similarity_search(query, k=k, filter=filter_expr)
            return [(d, 0.5) for d in docs]
        except Exception:
            return []


# ----------------------------------------------------------------------
# Main retriever node
# ----------------------------------------------------------------------
def policy_retriever_node(state: HospitalAgentState) -> dict:
    """
    Two-stage retrieval for 'policy':
      1. Vector search (recall) — with filter if department is valid.
      2. Reranker (precision) — with graceful fallback to vector scores.
    Sanitization is inlined per fragment.
    """
    query = _build_retrieval_query(state)
    if not query:
        return {"retrieved_context": [], "agent_status": {"policy_retriever": "empty_query"}}

    user_department = _resolve_department(state.get("user_department"))
    print(f"📥 Policy retrieval: '{query}' (dept={user_department or 'unfiltered'})")

    # ── Stage 1: Vector search (with filter if department is valid) ─────
    candidate_docs = []
    filter_used = "unfiltered"
    if user_department:
        try:
            filter_expr = {"department": {"$in": ["All", user_department]}}
            candidate_docs = vector_store.similarity_search(
                query, k=INITIAL_CANDIDATE_K, filter=filter_expr
            )
            if candidate_docs:
                filter_used = f"dept={user_department}"
        except Exception as e:
            print(f"⚠️ Filtered search failed: {e}")

    # Fallback to unfiltered if the filter produced nothing
    if not candidate_docs:
        if user_department:
            print("⚠️ Filtered search returned 0 — falling back to unfiltered.")
            filter_used = "unfiltered_fallback"
        try:
            candidate_docs = vector_store.similarity_search(
                query, k=INITIAL_CANDIDATE_K
            )
        except Exception as e:
            print(f"⚠️ Vector search failed: {e}")
            return {
                "retrieved_context": [],
                "agent_status": {"policy_retriever": "error:vector_search"},
            }

    if not candidate_docs:
        return {"retrieved_context": [], "agent_status": {"policy_retriever": "no_match"}}

    # ── Stage 2: Rerank (with fallback) ─────────────────────────────────
    used_reranker = False
    try:
        reranked_docs = reranker.compress_documents(
            documents=candidate_docs,
            query=query,
        )
        used_reranker = True
        normalized = [
            {
                "text": d.page_content,
                "metadata": d.metadata,
                "score": float(d.metadata.get("relevance_score", 0.0)),
            }
            for d in reranked_docs
        ]
    except Exception as e:
        print(f"⚠️ Rerank failed: {e}. Falling back to vector scores.")
        scored = _retrieve_with_scores(
            query, k=5,
            department=user_department if filter_used.startswith("dept=") else None,
        )
        if scored:
            normalized = [
                {"text": d.page_content, "metadata": d.metadata, "score": s}
                for d, s in scored
            ]
        else:
            normalized = [
                {"text": d.page_content, "metadata": d.metadata, "score": 0.5}
                for d in candidate_docs[:5]
            ]

    # ── Stage 3: Sanitize + build fragments ─────────────────────────────
    fragments = []
    seen_hashes = set()
    blocked = 0
    score_floor = MIN_RERANK_SCORE if used_reranker else MIN_RELEVANCE_SCORE

    for item in normalized:
        if item["score"] < score_floor:
            continue

        content = item["text"]
        h = hash(content[:200])
        if h in seen_hashes:
            continue
        seen_hashes.add(h)

        matched = _scan_for_injection(content)
        if matched:
            blocked += 1
            print(f"🚨 Indirect injection in chunk "
                  f"{item['metadata'].get('chunk_id')} — dropping this fragment only.")
            continue

        fragments.append({
            "source": "policy_retriever",
            "content": content,
            "metadata": {
                **item["metadata"],
                ("rerank_score" if used_reranker else "vector_score"): round(item["score"], 4),
            },
        })

    status = "ok" if fragments else "no_match"
    if blocked:
        status += f"|sanitized:{blocked}"
    if not used_reranker:
        status += "|rerank_fallback"
    if filter_used.endswith("fallback"):
        status += "|unfiltered"

    print(f"✅ Returning {len(fragments)} fragment(s) [{status}]")

    return {
        "retrieved_context": fragments,
        "agent_status": {"policy_retriever": status},
    }


print("✅ Policy Retriever defined (hardened, data-aligned).")

HR Salary Agent Node

In [ ]:
# Cell 14d: HR Salary Agent Node (Turso Cloud)
import re
import json
import turso_serverless
from langchain_core.messages import HumanMessage, SystemMessage

print("💼 Defining HR Salary Agent Node (Turso Cloud)...")

SALARY_ALLOWED_CLEARANCES = {"Executive", "Department Head"}

def _turso():
    return turso_serverless.connect(
        os.environ["TURSO_DATABASE_URL"],
        auth_token=os.environ["TURSO_AUTH_TOKEN"],
    )

def _query_salary_record(emp_id: str) -> dict | None:
    """Fetch one salary record from Turso and parse the nested allowances."""
    conn = _turso()
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM salaries WHERE emp_id = ?", (emp_id,))
        rows = cur.fetchall()
        if not rows:
            return None
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()

    record = dict(zip(cols, rows[0]))
    # Turso stores 'allowances' as a JSON string
    if isinstance(record.get("allowances"), str):
        try:
            record["allowances"] = json.loads(record["allowances"])
        except Exception:
            pass
    return record


def hr_salary_agent_node(state: HospitalAgentState) -> dict:
    messages = state.get("messages") or []
    user_clearance = state.get("user_clearance", "Unknown")

    # 1) Authorization gate
    if user_clearance not in SALARY_ALLOWED_CLEARANCES:
        print(f"🚫 Salary access denied for clearance '{user_clearance}'.")
        return {
            "retrieved_context": [{
                "source": "hr_salary_agent",
                "content": (
                    f"ACCESS DENIED: Clearance level '{user_clearance}' is not authorized "
                    f"to view salary records. Only Executive and Department Head may do so."
                ),
                "metadata": {"denied": True, "clearance": user_clearance},
            }],
            "agent_status": {"hr_salary_agent": "denied"},
        }

    # 2) Extract EMP-XXXXXX (6 digits)
    raw = _message_text(messages[-1]) if messages else ""
    emp_id_match = re.search(r"\bEMP-\d{6}\b", raw, re.IGNORECASE)
    emp_id = emp_id_match.group(0).upper() if emp_id_match else None

    # 3) LLM fallback extraction if regex misses
    if not emp_id:
        try:
            extractor = ChatOpenAI(
                model=CONFIG["cheap_llm_model"],
                api_key=os.environ["OPENROUTER_API_KEY"],
                base_url="https://openrouter.ai/api/v1",
                temperature=0,
            )
            resp = extractor.invoke([
                SystemMessage(content=(
                    "Extract the employee ID (format EMP-000000) from the message. "
                    "If none is present, reply with exactly 'NONE'."
                )),
                HumanMessage(content=raw),
            ])
            candidate = _message_text(resp).strip().upper()
            if re.fullmatch(r"EMP-\d{6}", candidate):
                emp_id = candidate
        except Exception as e:
            print(f"⚠️ emp_id extraction failed: {e}")

    if not emp_id:
        return {
            "retrieved_context": [{
                "source": "hr_salary_agent",
                "content": "I need an employee ID (e.g. EMP-000001) to look up salary information.",
                "metadata": {"missing": "emp_id"},
            }],
            "agent_status": {"hr_salary_agent": "missing_emp_id"},
        }

    # 4) Query Turso
    print(f"💼 Querying salary for {emp_id} (Turso)...")
    try:
        record = _query_salary_record(emp_id)
        if record is None:
            result = f"No salary record found for Employee ID {emp_id}."
        else:
            allowances = record.get("allowances") or {}
            allowance_str = ", ".join(f"{k}: {v}" for k, v in allowances.items())
            base = record.get("base_salary_usd", 0) or 0
            result = (
                f"Salary details for {emp_id}:\n"
                f"- Tier: {record.get('tier')}\n"
                f"- Base Salary: ${base:,}\n"
                f"- Bonus Eligible: {record.get('bonus_eligible')}\n"
                f"- Allowances: {allowance_str}"
            )
    except Exception as e:
        result = f"Error accessing salary data: {e}"

    return {
        "retrieved_context": [{
            "source": "hr_salary_agent",
            "content": result,
            "metadata": {"emp_id": emp_id},
        }],
        "agent_status": {"hr_salary_agent": "ok" if "No salary" not in result else "not_found"},
    }

Patient Agent

In [ ]:
# Cell 14e: Patient Records Agent Node (Cross-Database Python JOIN)
import re
import turso_serverless

print("🏥 Defining Patient Records Agent Node (Cross-DB Relational JOIN)...")

PATIENT_ALLOWED_CLEARANCES = {"Executive", "Department Head", "Physician", "Intern"}
PHI_FIELDS = ["diagnosis", "treatment_notes"]

def _turso_patients():
    return turso_serverless.connect(
        os.environ["TURSO_PATIENTS_URL"],
        auth_token=os.environ["TURSO_PATIENTS_TOKEN"],
    )

def _turso_staff():
    return turso_serverless.connect(
        os.environ["TURSO_DATABASE_URL"],
        auth_token=os.environ["TURSO_AUTH_TOKEN"],
    )

def _query_patient_record(patient_id: str) -> dict | None:
    """Fetch patient record from Patients DB, then JOIN with Staff DB in Python."""

    # 1. Fetch Patient
    conn_p = _turso_patients()
    try:
        cur = conn_p.cursor()
        cur.execute("SELECT * FROM patients WHERE patient_id = ?", (patient_id,))
        rows = cur.fetchall()
        if not rows:
            return None
        cols = [d[0] for d in cur.description]
        patient_record = dict(zip(cols, rows[0]))
    finally:
        conn_p.close()

    # 2. Fetch Physician (if assigned)
    physician_id = patient_record.get("primary_physician_id")
    if physician_id:
        conn_s = _turso_staff()
        try:
            cur_s = conn_s.cursor()
            cur_s.execute("SELECT name, department, role FROM staff WHERE emp_id = ?", (physician_id,))
            s_rows = cur_s.fetchall()
            if s_rows:
                s_cols = [d[0] for d in cur_s.description]
                physician_record = dict(zip(s_cols, s_rows[0]))
                # Merge physician data into the patient dictionary
                patient_record["physician_name"] = physician_record.get("name")
                patient_record["physician_dept"] = physician_record.get("department")
                patient_record["physician_role"] = physician_record.get("role")
        finally:
            conn_s.close()

    return patient_record


def patient_agent_node(state: HospitalAgentState) -> dict:
    messages = state.get("messages") or []
    user_clearance = state.get("user_clearance", "Unknown")

    # 1) Authorization gate
    if user_clearance not in PATIENT_ALLOWED_CLEARANCES:
        return {
            "retrieved_context": [{
                "source": "patient_agent",
                "content": f"ACCESS DENIED: clearance '{user_clearance}' cannot view patient records.",
                "metadata": {"denied": True},
            }],
            "agent_status": {"patient_agent": "denied"},
        }

    # 2) Extract PAT-XXXXXXX (7 digits)
    raw = _message_text(messages[-1]) if messages else ""
    pid_match = re.search(r"\bPAT-\d{7}\b", raw, re.IGNORECASE)
    patient_id = pid_match.group(0).upper() if pid_match else None

    if not patient_id:
        return {
            "retrieved_context": [{
                "source": "patient_agent",
                "content": "I need a patient ID (e.g. PAT-0000001) to look up records.",
                "metadata": {"missing": "patient_id"},
            }],
            "agent_status": {"patient_agent": "missing_patient_id"},
        }

    # 3) Query the Patients DB + Staff DB
    print(f"🏥 Querying patient {patient_id} + Physician details (Cross-DB)...")
    try:
        record = _query_patient_record(patient_id)
        if record is None:
            result = f"No patient found with ID {patient_id}."
        else:
            # 4) PHI Masking
            for phi in PHI_FIELDS:
                if phi in record and record[phi] is not None:
                    record[phi] = "[REDACTED_PHI]"

            charges = record.get("total_charges_usd", 0) or 0

            # Handle case where physician wasn't found in Staff DB
            phys_name = record.get('physician_name') or "Unknown"
            phys_dept = record.get('physician_dept') or "Unknown"
            phys_role = record.get('physician_role') or "Unknown"
            phys_id = record.get('primary_physician_id') or "None"

            result = (
                f"Patient details for {patient_id}:\n"
                f"- Name: {record.get('full_name')}\n"
                f"- DOB: {record.get('dob')}\n"
                f"- Admission Date: {record.get('admission_date')}\n"
                f"- Room: {record.get('room_number')}\n"
                f"- Billing Status: {record.get('billing_status')}\n"
                f"- Total Charges: ${charges:,}\n"
                f"- Diagnosis: {record.get('diagnosis')}\n"
                f"- Treatment Notes: {record.get('treatment_notes')}\n"
                f"- Assigned Physician: {phys_name} (ID: {phys_id}, Dept: {phys_dept}, Role: {phys_role})\n"
            )
    except Exception as e:
        result = f"Error accessing patient data: {e}"

    return {
        "retrieved_context": [{
            "source": "patient_agent",
            "content": result,
            "metadata": {"patient_id": patient_id},
        }],
        "agent_status": {"patient_agent": "ok" if "No patient" not in result else "not_found"},
    }

Staff Directory Agent Node

In [ ]:
# Cell 14f: Staff Directory Agent Node (Turso Cloud)
import re
import json as _json
import turso_serverless
from langchain_core.messages import HumanMessage, SystemMessage

print("👥 Defining Staff Directory Agent Node (Turso Cloud)...")

STAFF_CROSS_DEPT_CLEARANCES = {"Executive", "Department Head"}
STAFF_COLUMNS = "emp_id, name, role, department"

def _turso():
    return turso_serverless.connect(
        os.environ["TURSO_DATABASE_URL"],
        auth_token=os.environ["TURSO_AUTH_TOKEN"],
    )


def _query_staff_records(department=None, role=None, name=None, limit=15):
    clauses, params = [], []
    if department:
        clauses.append("department LIKE ?"); params.append(f"%{department}%")
    if role:
        clauses.append("role LIKE ?"); params.append(f"%{role}%")
    if name:
        clauses.append("name LIKE ?"); params.append(f"%{name}%")

    where = (" WHERE " + " AND ".join(clauses)) if clauses else ""
    sql = f"SELECT {STAFF_COLUMNS} FROM staff{where} LIMIT ?"
    params.append(int(limit))

    conn = _turso()
    try:
        # Explicitly create a cursor to access .description
        cur = conn.cursor()
        cur.execute(sql, tuple(params))
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]  # <--- Fix is here
    finally:
        conn.close()
    return [dict(zip(cols, r)) for r in rows]


def _query_staff_records_by_emp_id(emp_id: str):
    conn = _turso()
    try:
        cur = conn.cursor()
        cur.execute(
            f"SELECT {STAFF_COLUMNS} FROM staff WHERE emp_id = ?", (emp_id,)
        )
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]  # <--- Fix is here
    finally:
        conn.close()
    return [dict(zip(cols, r)) for r in rows]

def _normalize_emp_id(raw_id: str) -> str:
    if not raw_id:
        return ""
    match = re.search(r"EMP-(\d+)", raw_id.upper())
    return f"EMP-{int(match.group(1)):06d}" if match else raw_id.upper()


def _format_staff_table(records: list[dict]) -> str:
    if not records:
        return "No matching staff records found."
    lines = [
        "| emp_id | name | role | department |",
        "|---|---|---|---|",
    ]
    for r in records:
        lines.append(
            f"| {r.get('emp_id','')} | {r.get('name','')} | {r.get('role','')} | "
            f"{r.get('department','')} |"
        )
    return "\n".join(lines)


def staff_agent_node(state: HospitalAgentState) -> dict:
    messages = state.get("messages") or []
    user_clearance = state.get("user_clearance", "Unknown")
    user_department = state.get("user_department") or None
    raw_user_id = state.get("user_id") or ""
    user_id = _normalize_emp_id(raw_user_id)

    raw = _message_text(messages[-1]) if messages else ""

    # ── 1) LLM filter extraction ────────────────────────────────────
    department = role = name = None
    try:
        extractor = ChatOpenAI(
            model=CONFIG["cheap_llm_model"],
            api_key=os.environ["OPENROUTER_API_KEY"],
            base_url="https://openrouter.ai/api/v1",
            temperature=0,
        )
        resp = extractor.invoke([
            SystemMessage(content=(
                "Extract search filters from the staff-directory query. "
                "Reply with ONLY a JSON object with these keys (use null when absent):\n"
                '{"department": "...", "role": "...", "name": "..."}\n'
                "Do not add commentary or markdown."
            )),
            HumanMessage(content=raw),
        ])
        text = _message_text(resp).strip()
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end != -1:
            parsed = _json.loads(text[start:end + 1])
            department = parsed.get("department") or None
            role = parsed.get("role") or None
            name = parsed.get("name") or None
    except Exception as e:
        print(f"⚠️ Staff filter extraction failed: {e}")

    # ── 2) Department scoping ──────────────────────────────────────
    scoped_note = ""
    if user_clearance not in STAFF_CROSS_DEPT_CLEARANCES:
        if user_department:
            department = user_department
            scoped_note = f" (scoped to your department: {user_department})"
        elif user_id.startswith("EMP-"):
            print(f"👥 No department set — self-lookup for {user_id}")
            try:
                records = _query_staff_records_by_emp_id(user_id)
                result = _format_staff_table(records)
            except Exception as e:
                result = f"Error querying staff directory: {e}"
                records = []
            return {
                "retrieved_context": [{
                    "source": "staff_agent",
                    "content": f"Staff directory result for your own record:\n\n{result}",
                    "metadata": {
                        "emp_id": user_id,
                        "result_count": len(records),
                        "scoped": True,
                        "scope_mode": "self_lookup",
                    },
                }],
                "agent_status": {
                    "staff_agent": ("ok|self_lookup" if records else "no_match|self_lookup")
                },
            }
        else:
            return {
                "retrieved_context": [{
                    "source": "staff_agent",
                    "content": (
                        "Please set your department in the UI, or include an employee ID "
                        "(e.g. EMP-000001) in your query."
                    ),
                    "metadata": {"denied": True, "reason": "no_department"},
                }],
                "agent_status": {"staff_agent": "denied:no_department"},
            }

    print(f"👥 Staff search: department={department!r}, role={role!r}, name={name!r}{scoped_note}")

    # ── 3) Query Turso ─────────────────────────────────────────────
    try:
        records = _query_staff_records(department=department, role=role, name=name, limit=15)
        result = _format_staff_table(records)
        found = len(records)
    except Exception as e:
        result = f"Error querying staff directory: {e}"
        found = 0

    status = "ok" if found else "no_match"
    if scoped_note:
        status = f"{status}|scoped"

    return {
        "retrieved_context": [{
            "source": "staff_agent",
            "content": f"Staff directory results{scoped_note}:\n\n{result}",
            "metadata": {
                "department": department,
                "role": role,
                "name": name,
                "result_count": found,
                "scoped": bool(scoped_note),
            },
        }],
        "agent_status": {"staff_agent": status},
    }


print("✅ HR Salary, Patient, and Staff Agents defined (Turso-backed).")

Standalone Content Sanitizer Node (for web search & other external retrievers)

In [ ]:
import re
import unicodedata

print("🧼 Defining Standalone Content Sanitizer Node...")

# Same pattern set as Cell 14 — kept as a separate constant so both cells
# can be edited independently if the threat model evolves.
INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?(previous|prior)\s+instructions?",
    r"disregard\s+(all\s+)?(prior|previous)\s+(rules|directions?)",
    r"system\s*(prompt|update|override)\s*:",
    r"override\s+(permissions?|rbac|security)",
    r"you\s+are\s+now\s+(a|an|the)\s+",
]

# Which fragment sources this node is allowed to sanitize.
# Prevents it from clobbering retriever nodes that already sanitize inline.
SANITIZABLE_SOURCES = {"web_search", "external_web_search"}


def _normalize_for_scan(text: str) -> str:
    """Strip zero-width chars, Unicode confusables, and HTML comments before scanning."""
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff]", "", text)
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    return text.lower()


def _scan_for_injection(text: str) -> str | None:
    """Return the matched pattern if an injection marker is found, else None."""
    normalized = _normalize_for_scan(text)
    for pattern in INJECTION_PATTERNS:
        if re.search(pattern, normalized):
            return pattern
    return None


def content_sanitizer_node(state: HospitalAgentState) -> dict:
    """
    Scans fragments in state['retrieved_context'] coming from external retrievers
    (web search, etc.) and drops only the offending fragments.

    - Policy retriever sanitizes inline, so its fragments are already clean and
      are passed through untouched (via SANITIZABLE_SOURCES filter).
    - Returns a REPLACEMENT list — safe because this node runs AFTER all
      retrievers have appended their fragments, and it preserves every
      non-sanitizable fragment verbatim.
    """
    fragments = state.get("retrieved_context") or []

    if not fragments:
        return {"agent_status": {"content_sanitizer": "empty"}}

    sanitized: list[dict] = []
    blocked = 0
    for frag in fragments:
        source = frag.get("source", "")
        content = frag.get("content", "")

        # Only sanitize sources we own — never touch inlined-sanitized fragments
        if source not in SANITIZABLE_SOURCES:
            sanitized.append(frag)
            continue

        matched = _scan_for_injection(content)
        if matched:
            blocked += 1
            print(f"🚨 Injection in {source} fragment "
                  f"(chunk: {frag.get('metadata', {}).get('chunk_id', 'n/a')}, "
                  f"pattern: {matched}) — dropping this fragment only.")
            continue

        sanitized.append(frag)

    status = f"blocked:{blocked}" if blocked else "ok"
    print(f"🧼 Content sanitizer done: {len(fragments)} → {len(sanitized)} fragment(s) [{status}]")

    return {
        "retrieved_context": sanitized,   # full replacement — see docstring
        "agent_status": {"content_sanitizer": status},
    }


print("✅ Content Sanitizer node defined (targets web/external sources only).")

Web Search Retriever Node

In [ ]:
# Cell 14c: Web Search Retriever Node (Tavily-optimized)
import random
import re
import threading
import time
from collections import OrderedDict

print("🌐 Defining Web Search Retriever Node (Tavily)...")

# ─────────────────────────────────────────────────────────────────────────────
# Cache / throttle configurations
# ─────────────────────────────────────────────────────────────────────────────
WEB_CACHE_TTL_SECONDS = 60 * 60
WEB_NEG_CACHE_TTL_SECONDS = 5 * 60
WEB_CACHE_MAX_ENTRIES = 256
WEB_MIN_INTERVAL_SECONDS = 1.0     # Tavily handles rate limits well, reduced from 1.5
WEB_MAX_ATTEMPTS = 3
WEB_BACKOFF_BASE_SECONDS = 2.0
WEB_MAX_QUERY_CHARS = 400
WEB_MAX_RESULT_CHARS = 8000

_web_cache: "OrderedDict[str, tuple]" = OrderedDict()
_web_cache_lock = threading.Lock()
_web_rate_lock = threading.Lock()
_web_last_call = {"t": 0.0}


def _message_text(message) -> str:
    content = (
        message.get("content") if isinstance(message, dict)
        else getattr(message, "content", "")
    )
    if isinstance(content, list):
        content = " ".join(
            b.get("text", "") if isinstance(b, dict) else str(b) for b in content
        )
    return content if isinstance(content, str) else str(content or "")


def _cache_key(query: str) -> str:
    return query.casefold()


def _cache_get(key: str):
    with _web_cache_lock:
        entry = _web_cache.get(key)
        if entry is None:
            return None
        expires_at, status, text = entry
        if time.monotonic() >= expires_at:
            del _web_cache[key]
            return None
        _web_cache.move_to_end(key)
        return status, text


def _cache_put(key: str, status: str, text: str, ttl: float) -> None:
    with _web_cache_lock:
        _web_cache[key] = (time.monotonic() + ttl, status, text)
        _web_cache.move_to_end(key)
        while len(_web_cache) > WEB_CACHE_MAX_ENTRIES:
            _web_cache.popitem(last=False)


def _throttle() -> None:
    with _web_rate_lock:
        now = time.monotonic()
        wait = _web_last_call["t"] + WEB_MIN_INTERVAL_SECONDS - now
        _web_last_call["t"] = max(now, _web_last_call["t"] + WEB_MIN_INTERVAL_SECONDS)
    if wait > 0:
        time.sleep(wait)


def _is_rate_limit_error(err: Exception) -> bool:
    msg = str(err).lower()
    return any(k in msg for k in ("ratelimit", "rate limit", "429", "402"))


def _search_with_retries(query: str):
    last_error = None
    for attempt in range(1, WEB_MAX_ATTEMPTS + 1):
        _throttle()
        try:
            return external_web_search.invoke({"query": query}), None
        except Exception as e:
            last_error = e
            if attempt == WEB_MAX_ATTEMPTS:
                break
            delay = WEB_BACKOFF_BASE_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 1)
            print(f"⚠️ Web search attempt {attempt}/{WEB_MAX_ATTEMPTS} failed "
                  f"({str(e)[:120]}). Retrying in {delay:.1f}s...")
            time.sleep(delay)
    return None, last_error


def _build_fragment(query: str, content: str, cached: bool) -> dict:
    return {
        "source": "web_search",
        "content": content,
        "metadata": {
            "query": query,
            "title": f"Web results for: {query}",
            "category": "web",
            "department": "External",
            "header_path": "",
            "cached": cached,
        },
    }


# ─────────────────────────────────────────────────────────────────────────────
# Node
# ─────────────────────────────────────────────────────────────────────────────
def web_search_node(state: HospitalAgentState) -> dict:
    """
    Tavily-driven web search:
      1. Passes the raw user query to Tavily (no LLM rewrite needed).
      2. Caches and throttles the request.
      3. Appends the clean Tavily markdown fragment to retrieved_context.
    """
    try:
        messages = state.get("messages") or []
        if not messages:
            return {"retrieved_context": [], "agent_status": {"web_search": "empty_query"}}

        raw_question = _message_text(messages[-1]).strip()
        if not raw_question:
            return {"retrieved_context": [], "agent_status": {"web_search": "empty_query"}}

        # Clean up the query slightly
        search_query = re.sub(r"\s+", " ", raw_question).strip()
        print(f"🌐 Web search for query: '{search_query}'")

        # ── Step 1: Cache lookup ───────────────────────────────────────────
        key = _cache_key(search_query)
        hit = _cache_get(key)
        if hit is not None:
            status, text = hit
            if status == "no_match":
                print("💾 Cache hit (known empty result).")
                return {"retrieved_context": [], "agent_status": {"web_search": "no_match"}}
            print(f"💾 Cache hit — reusing {len(text)} chars.")
            return {
                "retrieved_context": [_build_fragment(search_query, text, cached=True)],
                "agent_status": {"web_search": "ok"},
            }

        # ── Step 2: Live search ────────────────────────────────────────────
        raw_result, error = _search_with_retries(search_query)
        if error is not None:
            print(f"⚠️ Web search failed after {WEB_MAX_ATTEMPTS} attempts: {error}")
            return {
                "retrieved_context": [],
                "agent_status": {"web_search": f"error:{str(error)[:200]}"},
            }

        if raw_result is None:
            raw_result = ""
        elif not isinstance(raw_result, str):
            raw_result = str(raw_result)
        raw_result = raw_result.strip()

        if not raw_result or "No relevant web results" in raw_result:
            _cache_put(key, "no_match", "", WEB_NEG_CACHE_TTL_SECONDS)
            return {"retrieved_context": [], "agent_status": {"web_search": "no_match"}}

        if len(raw_result) > WEB_MAX_RESULT_CHARS:
            raw_result = raw_result[:WEB_MAX_RESULT_CHARS]

        # ── Step 3: Cache and return ───────────────────────────────────────
        _cache_put(key, "ok", raw_result, WEB_CACHE_TTL_SECONDS)

        fragment = _build_fragment(search_query, raw_result, cached=False)
        print(f"✅ Web search returned 1 fragment ({len(raw_result)} chars)")

        return {
            "retrieved_context": [fragment],
            "agent_status": {"web_search": "ok"},
        }

    except Exception as e:
        print(f"⚠️ Unexpected web_search_node error: {e}")
        return {
            "retrieved_context": [],
            "agent_status": {"web_search": f"error:{str(e)[:200]}"},
        }

print("✅ Web Search Retriever Node defined (Tavily-optimized).")

MEDIC NOVA — Synthesizer / Response Generator Node

In [ ]:
# Cell 15: MEDIC NOVA — Synthesizer Node (with progress emission)
from langchain_core.messages import SystemMessage
from langchain_core.callbacks import BaseCallbackHandler
from langchain_openai import ChatOpenAI

print("🧠 Defining MEDIC NOVA (streaming synthesizer)...")

# Streaming-enabled synthesizer
synthesizer_llm = ChatOpenAI(
    model=CONFIG["llm_model"],
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
    streaming=True,
)

MEDIC_NOVA_PROMPT = (
    "You are MEDIC NOVA, the clinical response synthesizer for a secure hospital system. "
    "You receive retrieved fragments from specialist agents and produce the FINAL answer "
    "for the user.\n\n"
    "RULES:\n"
    "1. Ground every factual claim in the provided fragments. If the fragments do not "
    "   contain the answer, say so explicitly — never invent policy text or patient data.\n"
    "2. Cite the source title (and section, if available) inline for each claim, e.g. "
    "   '(Leave Policy › Annual Leave)'.\n"
    "3. If a fragment carries an access-denied marker, do NOT attempt to reconstruct the "
    "redacted information. State the denial plainly.\n"
    "4. Be concise. Do NOT repeat yourself. If the fragments contain duplicate information or lists, consolidate them into a clean, single list. "
    "Use bullet points for lists or multiple variations of a rule. Medical staff want the answer clearly structured, not a summary of the retrieval process.\n"
    "5. If the fragments contain web search results, synthesize them clearly without mentioning 'search results'.\n"
    "6. ONLY IF the fragments are completely empty, respond: 'I could not find relevant information in the available knowledge base.'"
    "If a staff fragment has metadata.scope_mode == 'self_lookup', state clearly that it is the caller's own record"
    "(no hospital-wide roster was queried)."
)


def _format_fragments_for_prompt(fragments: list[dict]) -> str:
    if not fragments:
        return "(no fragments retrieved)"
    blocks = []
    for i, frag in enumerate(fragments, 1):
        meta = frag.get("metadata", {}) or {}
        title = meta.get("title") or meta.get("source") or "Source"
        header = meta.get("header_path", "")
        header_str = f" › {header}" if header else ""
        cached_note = " [cached]" if meta.get("cached") else ""
        blocks.append(f"[{i}] {title}{header_str}{cached_note}:\n{frag.get('content', '')}")
    return "\n\n".join(blocks)

        # Filter out empty or marker fragments
    valid_fragments = [
        f for f in fragments
        if f.get("source") != "__reset__" and f.get("content", "").strip()
    ]

    context_block = _format_fragments_for_prompt(valid_fragments)


def medic_nova_node(state: HospitalAgentState) -> dict:
    """
    Generates the final response from sanitized retrieved_context.
    Single writer of final_output. Progress is emitted on the 'progress'
    stream channel so the UI can render 'Evaluating → Compiling → Generating'.
    """
    # Filter out the reset marker so MEDIC NOVA doesn't count it as a fragment
    raw_fragments = state.get("retrieved_context") or []
    fragments = [f for f in raw_fragments if f.get("source") != "__reset__"]
    user_clearance = state.get("user_clearance", "Unknown")
    intent = state.get("intent", "general")

    # These get picked up by stream_mode="custom" on the compiled app
    from langgraph.config import get_stream_writer
    writer = get_stream_writer()

    # ── Phase 1: Evaluate ────────────────────────────────────────
    writer({"phase": "evaluating",
            "detail": f"Reviewing {len(fragments)} fragment(s) for intent '{intent}'"})

    context_block = _format_fragments_for_prompt(fragments)

    # ── Phase 2: Compile ─────────────────────────────────────────
    writer({"phase": "compiling",
            "detail": f"Assembling answer from {len(fragments)} source(s)"})

    synthesis_messages = [
        SystemMessage(content=MEDIC_NOVA_PROMPT),
        SystemMessage(content=f"Caller clearance: {user_clearance}"),
        SystemMessage(content=f"Retrieved fragments:\n{context_block}"),
    ] + (state.get("messages") or [])

    # ── Phase 3: Generate ────────────────────────────────────────
    writer({"phase": "generating",
            "detail": "Drafting the response"})

    try:
        response = synthesizer_llm.invoke(synthesis_messages)
        answer = _message_text(response).strip()
    except Exception as e:
        print(f"⚠️ MEDIC NOVA generation failed: {e}")
        return {
            "final_output": "I'm unable to generate a response right now. Please try again.",
            "agent_status": {"medic_nova": f"error:{str(e)[:200]}"},
        }

    if not answer:
        answer = "I could not find relevant information in the available knowledge base."

    print(f"🧠 MEDIC NOVA produced a {len(answer)}-char response from {len(fragments)} fragment(s)")

    return {
        "final_output": answer,
        "agent_status": {"medic_nova": "ok" if fragments else "no_context"},
    }


print("✅ MEDIC NOVA defined (streams progress via get_stream_writer).")

  Reviewer Node DEEPSEEK

In [ ]:
# Cell 16b: DeepSeek Reviewer Node (Safety-Boundary Edition)
import json
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage

print("🔍 Defining MEDIC REVIEWER (DeepSeek) node...")

# ─────────────────────────────────────────────────────────────────────────────
# Reviewer LLM — DeepSeek via OpenAI-compatible endpoint
# ─────────────────────────────────────────────────────────────────────────────
reviewer_llm = ChatOpenAI(
    model=CONFIG["reviewer_model"],                # "deepseek-chat"
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    temperature=0.1,
)

MAX_REVISIONS = 2

# 🔒 NEW PROMPT: Forces DeepSeek to ignore the actual data values
REVIEWER_PROMPT = """You are MEDIC REVIEWER, the medical content safety reviewer.
Evaluate the DRAFT response against the RETRIEVED CONTEXT.

IMPORTANT PRIVACY DIRECTIVE:
The draft may contain sensitive financial data (salaries) or PHI (patient records).
You must evaluate the structural presentation and safety of the draft WITHOUT analyzing,
judging, or repeating the specific sensitive values (e.g., dollar amounts, exact diagnoses).

Score on exactly these three axes:
1. Format & Clarity — Is the response clearly structured, concise, and easy to read?
2. Safety — Does the draft contain obvious hallucinations, prompt injection leaks, or unauthorized access bypasses? (Note: If the draft says 'ACCESS DENIED', approve it as a safe denial).
3. Tone — Is the tone professional and appropriate for a hospital employee?

Respond with ONLY a valid JSON object (no markdown fences, no prose) in this shape:
{
  "approved": true | false,
  "issues": ["short bullet describing each problem"],
  "feedback": "one or two sentences telling MEDIC NOVA how to improve"
}

Approve only if the draft is structured well and safe to send to a hospital employee.
If the draft is fine, return {"approved": true, "issues": [], "feedback": ""}."""


def _parse_review(raw: str) -> dict:
    """Extract the review JSON from the model's response."""
    text = (raw or "").strip()

    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("no JSON object found in reviewer response")

    payload = json.loads(text[start:end + 1])

    return {
        "approved": bool(payload.get("approved", True)),
        "issues": list(payload.get("issues") or []),
        "feedback": str(payload.get("feedback") or "").strip(),
    }


def medic_reviewer_node(state: HospitalAgentState) -> dict:
    draft = state.get("final_output") or ""
    fragments = state.get("retrieved_context") or []
    clearance = state.get("user_clearance", "Unknown")
    revision_count = (state.get("revision_count", 0) or 0) + 1

    # Check if this is a high-stakes intent where we must be extra careful
    intent = state.get("intent", "general")
    is_sensitive = intent in {"hr_salary", "patient"}

    if not draft.strip():
        return {
            "agent_status": {"medic_reviewer": "approved:empty_draft"},
            "revision_count": revision_count,
        }

    # Build a compact context block from the fragments
    if fragments:
        ctx_block = "\n\n".join(
            f"[{i+1}] {f.get('metadata', {}).get('title', 'Source')}: {f['content']}"
            for i, f in enumerate(fragments[:5])
        )
    else:
        ctx_block = "(no fragments were retrieved)"

    review_messages = [
        SystemMessage(content=REVIEWER_PROMPT),
        SystemMessage(content=f"Caller clearance: {clearance}"),
        SystemMessage(content=f"Intent: {intent} (Sensitive: {is_sensitive})"),
        SystemMessage(content=f"Retrieved context:\n{ctx_block}"),
        SystemMessage(content=f"Draft response to review:\n{draft}"),
    ]

    try:
        response = reviewer_llm.invoke(review_messages)
        review = _parse_review(_message_text(response))
    except Exception as e:
        print(f"⚠️ MEDIC REVIEWER failed ({type(e).__name__}: {e}) — defaulting to approved.")
        return {
            "agent_status": {"medic_reviewer": f"approved:error:{str(e)[:120]}"},
            "revision_count": revision_count,
        }

    if review["approved"]:
        print("✅ MEDIC REVIEWER approved the draft.")
        return {
            "agent_status": {"medic_reviewer": "approved"},
            "revision_count": revision_count,
        }

    # Rejected — annotate the draft
    print(f"🚫 MEDIC REVIEWER rejected draft. Issues: {review['issues']}")
    annotated = (
        f"[REVISION NEEDED]\n"
        f"Issues: {'; '.join(review['issues']) or 'unspecified'}\n"
        f"Feedback: {review['feedback'] or 'please strengthen formatting'}\n\n"
        f"--- Previous draft ---\n{draft}"
    )
    return {
        "agent_status": {"medic_reviewer": "rejected"},
        "final_output": annotated,
        "revision_count": revision_count,
    }

print("✅ MEDIC REVIEWER node defined successfully (Privacy-Boundary).")

Conditional Edge — Escalate to Human Review?

In [ ]:

print("🔀 Defining human-review escalation router...")

# Intents that always require a human in the loop
HIGH_STAKES_INTENTS = {"hr_salary", "patient"}

# If the AI reviewer rejected, never send to human — loop back to MEDIC NOVA first
def route_after_reviewer(state: HospitalAgentState) -> str:
    """
    Decides the next hop after the DeepSeek reviewer:
      - rejected  -> "medic_nova"     (revise with feedback)
      - approved + high-stakes intent -> "human_review"
      - approved + low-stakes intent  -> "output_guardrail"
    """
    review_status = (state.get("agent_status") or {}).get("medic_reviewer", "")
    intent = state.get("intent", "general")

    if review_status == "rejected":
        print(f"↩️  Reviewer rejected — routing back to MEDIC NOVA for revision.")
        return "medic_nova"

    if intent in HIGH_STAKES_INTENTS:
        print(f"🧑‍⚕️ High-stakes intent '{intent}' — escalating to human review.")
        return "human_review"

    print(f"➡️  Low-stakes intent '{intent}' — skipping human review.")
    return "output_guardrail"

Human In Thte Loop

In [ ]:
def human_review_node(state: HospitalAgentState) -> dict:
    draft = state.get("final_output") or ""

    human_decision = interrupt({
        "action": "review_draft",
        "draft": draft,
        "instructions": "Approve, reject, or edit.",
    })

    if human_decision.get("action") == "approve":
        # If the user didn't edit the text, keep the original draft
        final_text = human_decision.get("edited_draft") or draft
        return {
            "final_output": final_text,
            "agent_status": {"human_review": "approved"},
        }
    elif human_decision.get("action") == "reject":
        return {
            "final_output": "Response withheld pending further review.",
            "agent_status": {"human_review": "rejected"},
        }
    else:
        final_text = human_decision.get("edited_draft") or draft
        return {
            "final_output": final_text,
            "agent_status": {"human_review": "edited"},
        }

Output Guardrail & PII Redaction Node

In [ ]:
# Cell 16: Output Guardrail & PII Redaction Node
import re

print("🛡️ Defining Output Guardrail & PII Redaction Node...")

# ─────────────────────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────────────────────
# When True: offending patterns are REDACTED in place (soft block).
# When False: the entire response is replaced with a generic refusal.
GUARDRAIL_REDACT_MODE = True

# On any leak/PII hit, prefix the response with a short warning so the user
# knows the answer was sanitized. Set to False for silent redaction.
GUARDRAIL_NOTICE_USER = True

# ─────────────────────────────────────────────────────────────────────────────
# Pattern sets
# ─────────────────────────────────────────────────────────────────────────────
# 1. System-prompt / infrastructure leakage
LEAK_PATTERNS = [
    (r"MEDIC[_ ](ASTRO|NOVA)\s+(PROMPT|SYSTEM|INSTRUCTIONS?)", "[REDACTED_LEAK]"),
    (r"SUPERVISOR_SYSTEM_PROMPT|MEDIC_ASTRO_PROMPT|MEDIC_NOVA_PROMPT", "[REDACTED_LEAK]"),
    (r"\bsk-[A-Za-z0-9]{20,}\b", "[REDACTED_API_KEY]"),                   # OpenAI-style
    (r"\bsk-or-v1-[A-Za-z0-9]{20,}\b", "[REDACTED_API_KEY]"),             # OpenRouter
    (r"\bpcsk_[A-Za-z0-9_]{20,}\b", "[REDACTED_API_KEY]"),                # Pinecone
    (r"PINECONE_API_KEY|OPENROUTER_API_KEY|DEEPSEEK_API_KEY", "[REDACTED_SECRET]"),
    (r"\b(?:tool|function)_(?:call|name)\s*[:=]\s*\w+", "[REDACTED_TOOL]"),
]

# 2. PHI / PII patterns
PHI_PATTERNS = [
    # US-style SSN
    (r"\b\d{3}-\d{2}-\d{4}\b", "[REDACTED_SSN]"),
    # MRN / patient IDs that look like identifiers (not generic numbers)
    (r"\bMRN[:\s#]*\d{6,10}\b", "[REDACTED_MRN]"),
    # Email addresses
    (r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", "[REDACTED_EMAIL]"),
    # Phone numbers (US/UK-ish — loosen if your region differs)
    (r"\b(?:\+?\d{1,3}[\s-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b", "[REDACTED_PHONE]"),
    # Dates that appear next to "DOB" / "born"
    (r"(?:DOB|born|date of birth)[:\s]*\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", "[REDACTED_DOB]"),
    # Explicit PHI re-introduction markers (rare but worth catching)
    (r"\[REDACTED_PHI\][^.\n]{0,60}(diagnos|treat|prescr)", "[REDACTED_PHI_CONTEXT]"),
]

# 3. Access-denied reconstruction markers — if the model ignored an access
#    denial and then continued to describe the withheld data, catch it.
DENIAL_PATTERNS = [
    (r"ACCESS\s+DENIED[^.\n]*\.?\s*(?:However|But|Although)[^.]*\.",
     "[REDACTED_DENIAL_RECONSTRUCTION]"),
]

ALL_PATTERNS = LEAK_PATTERNS + PHI_PATTERNS + DENIAL_PATTERNS

# Compiled regexes (compiled once at import for speed)
_COMPILED_PATTERNS = [(re.compile(p, re.IGNORECASE), repl) for p, repl in ALL_PATTERNS]


def _scan_and_redact(text: str) -> tuple[str, dict]:
    """
    Applies all pattern sets in sequence.
    Returns (sanitized_text, hit_summary) where hit_summary maps category → count.
    """
    hits = {"leak": 0, "phi": 0, "denial": 0}
    categories = [("leak", LEAK_PATTERNS), ("phi", PHI_PATTERNS), ("denial", DENIAL_PATTERNS)]

    sanitized = text
    for category, patterns in categories:
        for pattern, replacement in patterns:
            new_text, n = re.subn(pattern, replacement, sanitized, flags=re.IGNORECASE)
            if n > 0:
                hits[category] += n
                sanitized = new_text

    return sanitized, hits


# ─────────────────────────────────────────────────────────────────────────────
# Node
# ─────────────────────────────────────────────────────────────────────────────
def output_guardrail_node(state: HospitalAgentState) -> dict:
    """
    Final safety filter on final_output before it reaches the user.
    - Reads: final_output, agent_status (for context)
    - Writes: final_output (sanitized), agent_status["output_guardrail"]
    - Never touches: retrieved_context, messages, intent, is_safe
    """
    raw_output = state.get("final_output") or ""

    if not raw_output.strip():
        print("⚠️ Output guardrail received empty final_output.")
        return {
            "final_output": "I could not produce a valid response. Please rephrase your question.",
            "agent_status": {"output_guardrail": "empty_input"},
        }

    sanitized, hits = _scan_and_redact(raw_output)
    total_hits = sum(hits.values())

    if total_hits == 0:
        print("✅ Output passed guardrail verification.")
        return {
            "final_output": sanitized,
            "agent_status": {"output_guardrail": "ok"},
        }

    # Hits detected
    print(f"🚨 Output guardrail blocked {total_hits} pattern(s): {hits}")

    if not GUARDRAIL_REDACT_MODE:
        # Hard block: replace entire response
        safe_message = (
            "⚠️ The response was withheld because it contained information that could not "
            "be safely disclosed. Please rephrase your question or contact the appropriate "
            "department."
        )
        return {
            "final_output": safe_message,
            "agent_status": {
                "output_guardrail": f"hard_block:{total_hits}",
                **{f"hits_{k}": v for k, v in hits.items() if v},
            },
        }

    # Soft block: redact in place, optionally prepend a notice
    if GUARDRAIL_NOTICE_USER:
        categories_hit = [k for k, v in hits.items() if v]
        notice = (
            f"⚠️ Note: portions of this response were sanitized "
            f"({', '.join(categories_hit)})."
        )
        sanitized = f"{notice}\n\n{sanitized}"

    return {
        "final_output": sanitized,
        "agent_status": {
            "output_guardrail": f"redacted:{total_hits}",
            **{f"hits_{k}": v for k, v in hits.items() if v},
        },
    }


print("✅ Output Guardrail Node defined successfully!")

Graph Assembly + Conditional Routing + Persistent Checkpointer

In [ ]:
# Cell 19: Graph Assembly + Conditional Routing + Development-Safe Checkpointer
import sqlite3
import os
import uuid
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

print("🏗️ Assembling the Hospital Agent graph...")

# ─────────────────────────────────────────────────────────────────────────────
# 1. Conditional edge functions
# ─────────────────────────────────────────────────────────────────────────────
MAX_REVISIONS = 2

# Only these intents require human approval
SENSITIVE_INTENTS = {"hr_salary", "patient"}


def route_from_medic_astro(state: HospitalAgentState) -> str:
    intent = state.get("intent", "general")
    if not state.get("is_safe", True):
        return "end_blocked"

    mapping = {
        "policy":     "policy_retriever",
        "hr_salary":  "hr_salary_agent",
        "staff":      "staff_agent",
        "patient":    "patient_agent",
        "web_search": "web_search_agent",
        "general":    "medic_nova",
        "blocked":    "end_blocked",
    }
    return mapping.get(intent, "medic_nova")


def route_after_guardrail(state: HospitalAgentState) -> str:
    """If the input guardrail blocked the request, skip straight to END."""
    if not state.get("is_safe", True):
        return "end_blocked"
    return "medic_astro"


def route_after_reviewer(state: HospitalAgentState) -> str:
    """DeepSeek reviewer decision."""
    status = (state.get("agent_status") or {}).get("medic_reviewer", "")
    intent = state.get("intent", "general")
    revisions = state.get("revision_count", 0) or 0
    agent_status = state.get("agent_status") or {}

    if status == "rejected":
        if revisions >= MAX_REVISIONS:
            print(f"⚠️ Max revisions ({MAX_REVISIONS}) reached — forcing through.")
            return "output_guardrail"
        return "medic_nova"

    # HITL check: Only escalate to human review for the specific sensitive intents
    touched_sensitive = (
        intent in SENSITIVE_INTENTS or
        "hr_salary_agent" in agent_status or
        "patient_agent" in agent_status
    )

    if touched_sensitive:
        print(f"🧑‍⚕️ Sensitive data detected (Intent: {intent}) — escalating to human review.")
        return "human_review"

    return "output_guardrail"


def route_after_human(state: HospitalAgentState) -> str:
    """Check if the human rejected the draft. If so, loop back to MEDIC NOVA."""
    status = (state.get("agent_status") or {}).get("human_review", "")
    if status == "rejected":
        print("🚫 Human rejected the draft — looping back to MEDIC NOVA.")
        return "medic_nova"
    return "output_guardrail"


# ─────────────────────────────────────────────────────────────────────────────
# 2. Build the graph
# ─────────────────────────────────────────────────────────────────────────────
graph = StateGraph(HospitalAgentState)

# --- Nodes ---------------------------------------------------------------
graph.add_node("input_guardrail",   input_guardrail_node)
graph.add_node("medic_astro",       medic_astro_node)
graph.add_node("policy_retriever",  policy_retriever_node)
graph.add_node("web_search_agent",  web_search_node)
graph.add_node("content_sanitizer", content_sanitizer_node)
graph.add_node("medic_nova",        medic_nova_node)
graph.add_node("medic_reviewer",    medic_reviewer_node)
graph.add_node("human_review",      human_review_node)
graph.add_node("output_guardrail",  output_guardrail_node)
graph.add_node("staff_agent",       staff_agent_node)
graph.add_node("hr_salary_agent",   hr_salary_agent_node)
graph.add_node("patient_agent",     patient_agent_node)

# --- Entry point ---------------------------------------------------------
graph.set_entry_point("input_guardrail")

# --- Guardrail → MEDIC ASTRO --------------------------------------------
graph.add_conditional_edges(
    "input_guardrail",
    route_after_guardrail,
    {
        "medic_astro": "medic_astro",
        "end_blocked": END,
    },
)

# --- MEDIC ASTRO → conditional specialist routing ------------------------
graph.add_conditional_edges(
    "medic_astro",
    route_from_medic_astro,
    {
        "policy_retriever":  "policy_retriever",
        "hr_salary_agent":   "hr_salary_agent",
        "staff_agent":       "staff_agent",
        "patient_agent":     "patient_agent",
        "web_search_agent":  "web_search_agent",
        "medic_nova":        "medic_nova",
        "end_blocked":       END,
    },
)

# --- Specialist → synthesizer --------------------------------------------
graph.add_edge("policy_retriever", "medic_nova")
graph.add_edge("hr_salary_agent",  "medic_nova")
graph.add_edge("patient_agent",    "medic_nova")
graph.add_edge("web_search_agent", "content_sanitizer")
graph.add_edge("content_sanitizer", "medic_nova")
graph.add_edge("staff_agent",      "medic_nova")

# --- Synthesizer → reviewer → conditional --------------------------------
graph.add_edge("medic_nova", "medic_reviewer")
graph.add_conditional_edges(
    "medic_reviewer",
    route_after_reviewer,
    {
        "medic_nova":       "medic_nova",
        "human_review":     "human_review",
        "output_guardrail": "output_guardrail",
    },
)

# --- Human review → check if rejected → output guardrail ----------------
graph.add_conditional_edges(
    "human_review",
    route_after_human,
    {
        "medic_nova":       "medic_nova",
        "output_guardrail": "output_guardrail",
    },
)

graph.add_edge("output_guardrail", END)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Checkpointer (DEV_MODE-aware)
# ─────────────────────────────────────────────────────────────────────────────
DEV_MODE = True   # ← flip to False only when schema + graph are frozen

if DEV_MODE:
    session_tag = uuid.uuid4().hex[:6]
    CHECKPOINT_DB_PATH = f"/tmp/hospital_agent_dev_{session_tag}.sqlite"
    print(f"🧪 DEV_MODE: ephemeral checkpointer at {CHECKPOINT_DB_PATH}")
else:
    CHECKPOINT_DB_PATH = os.path.join(DATA_DIR, "hospital_agent_checkpoints.sqlite")
    print(f"🏭 PROD_MODE: persistent checkpointer at {CHECKPOINT_DB_PATH}")

_ckpt_conn = sqlite3.connect(CHECKPOINT_DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(_ckpt_conn)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Compile
# ─────────────────────────────────────────────────────────────────────────────
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print(f"   - Nodes: 12")
print(f"   - Checkpointer: {'ephemeral (dev)' if DEV_MODE else 'persistent (prod)'}")

In [ ]:
!pip install -U --quiet langgraph-checkpoint-sqlite

Checkpointer utilities (inspect, reset, clean Drive)

In [ ]:
# Cell 19b: # Cell 19b: Checkpointer utilities (inspect, reset, clean Drive)
import os

def inspect_checkpointer():
    """Show what threads exist and how many checkpoints each has."""
    if not os.path.exists(CHECKPOINT_DB_PATH):
        print("(no checkpointer file yet)")
        return
    conn = sqlite3.connect(CHECKPOINT_DB_PATH)
    try:
        rows = conn.execute(
            "SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id"
        ).fetchall()
        if not rows:
            print("(checkpointer file exists but has no threads)")
            return
        print(f"📂 Active threads in {CHECKPOINT_DB_PATH}:")
        for tid, count in rows:
            print(f"   • {tid}: {count} checkpoint(s)")
    except Exception as e:
        print(f"⚠️ Inspect failed: {e}")
    finally:
        conn.close()


def reset_checkpointer():
    """Delete all checkpoints from the current DB. Use when testing gets confusing."""
    global checkpointer, _ckpt_conn
    try:
        _ckpt_conn.close()
    except Exception:
        pass
    if os.path.exists(CHECKPOINT_DB_PATH):
        os.remove(CHECKPOINT_DB_PATH)
        print(f"🗑️ Removed {CHECKPOINT_DB_PATH}")
    # Recreate the checkpointer so `app` still works
    _ckpt_conn = sqlite3.connect(CHECKPOINT_DB_PATH, check_same_thread=False)
    checkpointer = SqliteSaver(_ckpt_conn)
    print("✅ Checkpointer reset.")


def purge_drive_checkpoints():
    """Only relevant in PROD_MODE — deletes the persistent DB on Drive."""
    drive_db = os.path.join(DATA_DIR, "hospital_agent_checkpoints.sqlite")
    if os.path.exists(drive_db):
        size_mb = os.path.getsize(drive_db) / (1024 * 1024)
        os.remove(drive_db)
        print(f"🗑️ Deleted Drive checkpointer ({size_mb:.1f} MB): {drive_db}")
    else:
        print("(no Drive checkpointer to delete)")


print("✅ Checkpointer utilities ready:")
print("   inspect_checkpointer()   → show active threads")
print("   reset_checkpointer()     → wipe current session's state")
print("   purge_drive_checkpoints() → remove stale Drive DB (prod mode)")
import os

def inspect_checkpointer():
    """Show what threads exist and how many checkpoints each has."""
    if not os.path.exists(CHECKPOINT_DB_PATH):
        print("(no checkpointer file yet)")
        return
    conn = sqlite3.connect(CHECKPOINT_DB_PATH)
    try:
        rows = conn.execute(
            "SELECT thread_id, COUNT(*) FROM checkpoints GROUP BY thread_id"
        ).fetchall()
        if not rows:
            print("(checkpointer file exists but has no threads)")
            return
        print(f"📂 Active threads in {CHECKPOINT_DB_PATH}:")
        for tid, count in rows:
            print(f"   • {tid}: {count} checkpoint(s)")
    except Exception as e:
        print(f"⚠️ Inspect failed: {e}")
    finally:
        conn.close()


def reset_checkpointer():
    """Delete all checkpoints from the current DB. Use when testing gets confusing."""
    global checkpointer, _ckpt_conn
    try:
        _ckpt_conn.close()
    except Exception:
        pass
    if os.path.exists(CHECKPOINT_DB_PATH):
        os.remove(CHECKPOINT_DB_PATH)
        print(f"🗑️ Removed {CHECKPOINT_DB_PATH}")
    # Recreate the checkpointer so `app` still works
    _ckpt_conn = sqlite3.connect(CHECKPOINT_DB_PATH, check_same_thread=False)
    checkpointer = SqliteSaver(_ckpt_conn)
    print("✅ Checkpointer reset.")


def purge_drive_checkpoints():
    """Only relevant in PROD_MODE — deletes the persistent DB on Drive."""
    drive_db = os.path.join(DATA_DIR, "hospital_agent_checkpoints.sqlite")
    if os.path.exists(drive_db):
        size_mb = os.path.getsize(drive_db) / (1024 * 1024)
        os.remove(drive_db)
        print(f"🗑️ Deleted Drive checkpointer ({size_mb:.1f} MB): {drive_db}")
    else:
        print("(no Drive checkpointer to delete)")


print("✅ Checkpointer utilities ready:")
print("   inspect_checkpointer()   → show active threads")
print("   reset_checkpointer()     → wipe current session's state")
print("   purge_drive_checkpoints() → remove stale Drive DB (prod mode)")

Visualize the Compiled Graph Structure

In [ ]:
# Cell 19b: Visualize the Compiled Graph Structure
import os
from IPython.display import Image, display, Markdown

print("🗺️  Inspecting compiled graph structure...\n")

# ─────────────────────────────────────────────────────────────────────────────
# 1. ASCII diagram (works everywhere, no dependencies)
# ─────────────────────────────────────────────────────────────────────────────
try:
    ascii_diagram = app.get_graph().draw_ascii()
    print("── ASCII Graph ────────────────────────────────────────────")
    print(ascii_diagram)
    print("───────────────────────────────────────────────────────────\n")
except Exception as e:
    print(f"⚠️ ASCII rendering failed: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mermaid PNG (rendered inline in Colab)
# ─────────────────────────────────────────────────────────────────────────────
try:
    png_bytes = app.get_graph().draw_mermaid_png()
    out_path = os.path.join(DATA_DIR, "hospital_agent_graph.png")
    with open(out_path, "wb") as f:
        f.write(png_bytes)

    print(f"💾 Graph PNG saved to: {out_path}")
    display(Image(png_bytes))
except Exception as e:
    print(f"⚠️ PNG rendering failed: {e}")
    print("   Falling back to Mermaid source (paste into https://mermaid.live):\n")
    try:
        print(app.get_graph().draw_mermaid())
    except Exception as e2:
        print(f"⚠️ Mermaid source also failed: {e2}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Adjacency summary (node → outgoing edges)
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Adjacency Summary ──────────────────────────────────────")
try:
    g = app.get_graph()
    # Group edges by source node
    edges_by_source: dict[str, list[str]] = {}
    for edge in g.edges:
        src, dst = edge.source, edge.target
        edges_by_source.setdefault(src, []).append(dst)

    for src in sorted(edges_by_source):
        targets = sorted(set(edges_by_source[src]))
        arrow = ", ".join(targets)
        print(f"  {src:<22} → {arrow}")
except Exception as e:
    print(f"⚠️ Adjacency summary failed: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Node count sanity check
# ─────────────────────────────────────────────────────────────────────────────
print("\n── Node Inventory ─────────────────────────────────────────")
try:
    nodes = sorted(app.get_graph().nodes.keys())
    # Filter LangGraph's synthetic bookkeeping nodes
    real_nodes = [n for n in nodes if not n.startswith("__")]
    print(f"  Total nodes: {len(real_nodes)}")
    for n in real_nodes:
        print(f"    • {n}")
except Exception as e:
    print(f"⚠️ Node inventory failed: {e}")

SMOKE TEST

In [ ]:
from langchain_core.messages import HumanMessage

# Use a highly specific query to test the Pinecone retrieval and reranker
initial = {
    "messages": [HumanMessage(content="What is the exact procedure for emergency escalation regarding code protocols? Include the specific notification times and who must be notified.")],
    "user_id": "EMP-000009",
    "user_clearance": "Intern",
    "user_department": "Emergency", # Scoping to Emergency to match the policy data
    "retrieved_context": [{"source": "__reset__", "content": "", "metadata": {}}],
    "agent_status": {},
    "revision_count": 0,
}

config = {"configurable": {"thread_id": "smoke-test-policy-002"}}

print("🚀 Running Policy Test...\n")

# Stream the execution
for event in app.stream(initial, config, stream_mode="updates"):
    for node_name, update in event.items():
        # Print agent status updates
        status = (update.get("agent_status") or {}) if isinstance(update, dict) else {}
        if status:
            print(f"  ▸ {node_name}: {status}")

# After the stream finishes, pull the authoritative final state
snapshot = app.get_state(config)
final_output = (snapshot.values or {}).get("final_output", "(No output generated)")
final_context = (snapshot.values or {}).get("retrieved_context", [])

print("\n" + "="*50)
print(f"📚 FRAGMENTS RETRIEVED: {len(final_context)}")
print("="*50)
print("📝 FINAL OUTPUT:")
print(final_output)
print("="*50)

One test after wiring

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import interrupt, Command

# Quick patch to fix the missing global import in the environment
import sys
import __main__
__main__.interrupt = interrupt

initial = {
    "messages": [HumanMessage(content="What is the salary for EMP-000001?")],
    "user_id": "EMP-0050",
    "user_clearance": "Intern",
    "user_department": "Pediatrics",
    "retrieved_context": [],
    "agent_status": {},
    "revision_count": 0,
}

config = {"configurable": {"thread_id": "staff-test-2"}}
final_state = None

try:
    for event in app.stream(initial, config, stream_mode="updates"):
        for node, upd in event.items():
            if isinstance(upd, dict) and upd.get("agent_status"):
                print(f"{node} → {upd['agent_status']}")
            # Capture any field that carries the running draft / final answer
            if isinstance(upd, dict):
                final_state = {**(final_state or {}), **upd}
except Exception as e:
    print(f"\u26a0\ufe0f Error encountered: {e}")
    print("Note: If the error persists, please re-execute the cell defining human_review_node with 'from langgraph.types import interrupt' loaded.")

# After streaming, pull the authoritative final state from the checkpointer
snapshot = app.get_state(config)
final_output = (snapshot.values or {}).get("final_output", "(empty)")
print("\n──── FINAL OUTPUT ────")
print(final_output)

TEST 3

In [ ]:
from langchain_core.messages import HumanMessage

# 1. Setup the query
test_query = "Where is patient PAT-0000001 and who is their doctor?"

test_state = {
    "messages": [HumanMessage(content=test_query)],
    "user_id": "EMP-000009",
    "user_clearance": "Physician",
    "user_department": "Cardiology",
    "retrieved_context": [{"source": "__reset__", "content": "", "metadata": {}}],
    "agent_status": {},
    "revision_count": 0,
}

# 2. Use a unique thread ID for this test
config = {"configurable": {"thread_id": "hitl-test-999"}}

print(f"⌖ Testing Query: '{test_query}'")
print("-" * 50)

was_paused = False

# 3. Stream through the graph
for event in app.stream(test_state, config, stream_mode="updates"):
    for node_name, update in event.items():

        # Catch the Human Review Interrupt
        if node_name == "__interrupt__":
            was_paused = True
            interrupts = update if isinstance(update, (list, tuple)) else [update]
            payload = getattr(interrupts[0], "value", interrupts[0]) or {}
            print("\n⌒ GRAPH PAUSED! Human Review Required.")
            print(f"⌕ DRAFT TO REVIEW:\n{payload.get('draft', '')}")
            continue

        # Print node status
        status = (update.get("agent_status") or {}) if isinstance(update, dict) else {}
        if status:
            print(f"  ┘ {node_name}: {status}")

# 4. Check final state
if not was_paused:
    snapshot = app.get_state(config)
    final_output = (snapshot.values or {}).get("final_output", "(empty)")
    print("\n✅ Graph finished without pausing.")
    print(f"──── FINAL OUTPUT ────\n{final_output}")
else:
    print("\n" + "="*50)
    print("⌗ The graph is currently paused and waiting for you.")
    print("⌗ To resume and approve it, run this code:")
    # Doubled the inner curly braces to fix f-string interpretation error
    print(f"""
from langgraph.types import Command

resume_config = {{"configurable": {{"thread_id": "hitl-test-999"}}}}

# Notice we pass None for edited_draft, so it keeps the original text
print("┖ Resuming graph with approval...")
final_state = app.invoke(
    Command(resume={{"action": "approve", "edited_draft": None}}),
    resume_config
)

print("──── FINAL OUTPUT ────")
print(final_state.get("final_output"))
""")

In [ ]:
from langgraph.types import Command

resume_config = {"configurable": {"thread_id": "hitl-test-999"}}

# Notice we pass None for edited_draft, so it keeps the original text
print("▶️ Resuming graph with approval...")
final_state = app.invoke(
    Command(resume={"action": "approve", "edited_draft": None}),
    resume_config
)

print(f"──── FINAL OUTPUT ────")
print(final_state.get("final_output"))

TEST 4

In [ ]:
from langchain_core.messages import HumanMessage

# Use a new thread_id so the checkpointer doesn't mix it with previous tests
config = {"configurable": {"thread_id": "web-test-1"}}

initial_state = {
    "messages": [HumanMessage(content="What are the latest WHO guidelines for sepsis management?")],
    "user_id": "EMP-000009",
    "user_clearance": "Physician",
    "user_department": "Emergency",
    "retrieved_context": [{"source": "__reset__", "content": "", "metadata": {}}],
    "agent_status": {},
    "revision_count": 0,
}

print("🌐 Running Web Search through the full graph...")
for event in app.stream(initial_state, config, stream_mode="updates"):
    for node, upd in event.items():
        if isinstance(upd, dict) and upd.get("agent_status"):
            print(f"{node} → {upd['agent_status']}")

# Pull final answer from state
snapshot = app.get_state(config)
final_output = (snapshot.values or {}).get("final_output", "(empty)")
print("\n──── FINAL OUTPUT ────")
print(final_output)

Gradio Interface for MEDIC ASTRO Hospital Agent

In [ ]:
# Cell 20: Gradio Interface (OpenRouter Dark Theme + Audio + HITL)
!pip install -U --quiet gradio openai

import uuid
import os
import gradio as gr
import openai
from langchain_core.messages import HumanMessage
from langgraph.types import Command

print("🎨 Building Gradio interface (Custom Dark Theme)...")

whisper_client = openai.OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

PHASE_LABELS = {
    "evaluating": "🔍 Evaluating your query…",
    "compiling":  "🧩 Compiling sources…",
    "generating": "✍️ Generating answer…",
}

# ─────────────────────────────────────────────────────────────────────────────
# Session helpers
# ─────────────────────────────────────────────────────────────────────────────
def _new_thread_id(user_id: str) -> str:
    return f"{user_id}-{uuid.uuid4().hex[:8]}"

def _graph_config(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}}

def _status_line(agent_status: dict) -> str:
    if not agent_status:
        return "_no agent activity yet_"
    # Format like OpenRouter's telemetry: `node` → status
    parts = [f"`{k}` → {v}" for k, v in agent_status.items()]
    return "  ·  ".join(parts)

# ─────────────────────────────────────────────────────────────────────────────
# Streaming chat handler
# ─────────────────────────────────────────────────────────────────────────────
def respond(message, history, user_id, clearance, department, thread_state):
    message = (message or "").strip()
    if not message:
        yield history, "", thread_state, "_empty input_", ""
        return

    if not thread_state:
        thread_state = _new_thread_id(user_id)

    config = _graph_config(thread_state)

    initial_state = {
        "messages": [HumanMessage(content=message)],
        "user_id": user_id,
        "user_clearance": clearance,
        "user_department": department,
        "retrieved_context": [{"source": "__reset__", "content": "", "metadata": {}}],
        "agent_status": {},
        "revision_count": 0,
    }

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": ""},
    ]

    status_line = "⏳ Starting…"
    partial = ""
    review_draft = ""
    was_paused = False

    try:
        for mode, chunk in app.stream(
            initial_state, config, stream_mode=["custom", "messages", "updates"]
        ):
            if mode == "custom" and isinstance(chunk, dict):
                phase = chunk.get("phase", "")
                detail = chunk.get("detail", "")
                label = PHASE_LABELS.get(phase, phase)
                status_line = f"{label} {detail}".strip()
                yield history, "", thread_state, status_line, ""

            elif mode == "messages":
                msg_chunk, meta = chunk
                node = (meta or {}).get("langgraph_node", "")
                if node == "medic_nova":
                    token = getattr(msg_chunk, "content", "") or ""
                    if token:
                        partial += token
                        history[-1] = {"role": "assistant", "content": partial + "▍"}
                        yield history, "", thread_state, status_line, ""

            elif mode == "updates":
                for node_name, update in chunk.items():
                    # Catch the LangGraph interrupt properly
                    if node_name == "__interrupt__":
                        interrupts = update if isinstance(update, (list, tuple)) else [update]
                        if interrupts:
                            payload = getattr(interrupts[0], "value", interrupts[0]) or {}
                            review_draft = payload.get("draft", "")
                            was_paused = True
                            history[-1] = {
                                "role": "assistant",
                                "content": f"⏸️ *Awaiting human review:*\n\n{review_draft}",
                            }
                            status_line = "⏸️ Awaiting human review"
                            yield history, "", thread_state, status_line, review_draft
                        continue

                    if node_name.startswith("__"):
                        continue

                    if isinstance(update, dict):
                        st = update.get("agent_status") or {}
                        if st:
                            key = next(iter(st))
                            status_line = f"✓ {key}: {st[key]}"
                            yield history, "", thread_state, status_line, ""

    except Exception as e:
        history[-1] = {"role": "assistant", "content": f"⚠️ System error: {e}"}
        yield history, "", thread_state, f"_error: {str(e)[:120]}_", ""
        return

    if was_paused:
        yield history, "", thread_state, "⏸️ Awaiting human review", review_draft
        return

    final_text = partial or "(no response generated)"
    history[-1] = {"role": "assistant", "content": final_text}

    try:
        snapshot = app.get_state(config)
        final_status = (snapshot.values or {}).get("agent_status") or {}
        status_line = _status_line(final_status) or "✅ Complete"
    except Exception:
        status_line = "✅ Complete"

    yield history, "", thread_state, status_line, ""

# ─────────────────────────────────────────────────────────────────────────────
# Audio Transcription Handler
# ─────────────────────────────────────────────────────────────────────────────
def transcribe_audio(audio_path, history, user_id, clearance, department, thread_state):
    if not audio_path:
        yield history, "", thread_state, "_No audio detected_", ""
        return

    try:
        with open(audio_path, "rb") as audio_file:
            transcript = whisper_client.audio.transcriptions.create(
                model="openai/whisper-1",
                file=audio_file
            )
        user_message = transcript.text
    except Exception as e:
        yield history, "", thread_state, f"_Audio error: {e}_", ""
        return

    yield from respond(user_message, history, user_id, clearance, department, thread_state)

# ─────────────────────────────────────────────────────────────────────────────
# Human review handlers
# ─────────────────────────────────────────────────────────────────────────────
def _resume_review(payload: dict, thread_state: str, history):
    if not thread_state:
        yield [], "", thread_state, "_no active review_", ""
        return

    config = _graph_config(thread_state)
    partial = ""
    status_line = "⏳ Resuming…"

    try:
        for mode, chunk in app.stream(
            Command(resume=payload), config,
            stream_mode=["custom", "messages", "updates"],
        ):
            if mode == "custom" and isinstance(chunk, dict):
                label = PHASE_LABELS.get(chunk.get("phase", ""), chunk.get("phase", ""))
                status_line = f"{label} {chunk.get('detail', '')}".strip()
                yield history, "", thread_state, status_line, ""

            elif mode == "messages":
                msg_chunk, meta = chunk
                if (meta or {}).get("langgraph_node") == "medic_nova":
                    token = getattr(msg_chunk, "content", "") or ""
                    if token:
                        partial += token
                        history[-1] = {"role": "assistant", "content": f"✅ *Reviewed:* {partial}▍"}
                        yield history, "", thread_state, status_line, ""

            elif mode == "updates":
                for node_name, update in chunk.items():
                    if node_name.startswith("__"):
                        continue
                    if isinstance(update, dict):
                        st = update.get("agent_status") or {}
                        if st:
                            key = next(iter(st))
                            status_line = f"✓ {key}: {st[key]}"
                            yield history, "", thread_state, status_line, ""

    except Exception as e:
        history[-1] = {"role": "assistant", "content": f"⚠️ Resume error: {e}"}
        yield history, "", thread_state, f"_resume error: {str(e)[:120]}_", ""
        return

    # 🔥 THE CRITICAL FIX: Pull the authoritative state from the checkpointer
    snapshot = app.get_state(config)
    final_output = (snapshot.values or {}).get("final_output", "")

    # If the stream didn't yield any tokens (because it bypassed MEDIC NOVA),
    # use the final_output from the state.
    final_text = partial or final_output or "(no response generated)"

    history[-1] = {"role": "assistant", "content": f"✅ *Reviewed response:*\n\n{final_text}"}

    try:
        final_status = (snapshot.values or {}).get("agent_status") or {}
        status_line = _status_line(final_status) or "✅ Complete"
    except Exception:
        status_line = "✅ Complete"

    yield history, "", thread_state, status_line, ""

def approve_review(edited_text, thread_state, history):
    payload = {"action": "approve", "edited_draft": edited_text}
    yield from _resume_review(payload, thread_state, history)

def reject_review(edited_text, thread_state, history):
    payload = {"action": "reject", "edited_draft": edited_text}
    yield from _resume_review(payload, thread_state, history)

def reset_session():
    return [], "", None, "_new session_", ""


# ─────────────────────────────────────────────────────────────────────────────
# THEME & CSS (OpenRouter Dark Mode Emulation)
# ─────────────────────────────────────────────────────────────────────────────
# 1. Custom Theme definition
custom_theme = gr.themes.Soft(
    primary_hue="teal",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "ui-monospace", "monospace"],
).set(
    body_background_fill="#0f1115",            # Deep dark background
    body_background_fill_dark="#0f1115",
    block_background_fill="#1a1d24",           # Slightly lighter panels
    block_border_color="#2d333b",              # Subtle borders
    block_title_text_color="#8b949e",          # Muted titles
    body_text_color="#c9d1d9",                 # Off-white text
    body_text_color_subdued="#8b949e",
    button_primary_background_fill="#238636",  # GitHub green for Send
    button_primary_background_fill_hover="#2ea043",
    button_primary_text_color="#ffffff",
    button_secondary_background_fill="#21262d",# Dark grey for secondary buttons
    button_secondary_text_color="#c9d1d9",
    input_background_fill="#0d1117",           # Darker inputs
    input_border_color="#30363d",
)

# 2. Custom CSS for extra polish
CSS = """
/* Tighten up the layout */
.gradio-container { max-width: 1400px !important; margin: 0 auto; padding: 20px; }
footer { display: none !important; }

/* Chatbot specific styling */
.message-wrap { border-radius: 8px; }
.message.user { background-color: #2d333b !important; color: #fff !important; }
.message.bot { background-color: #161b22 !important; border: 1px solid #30363d; color: #c9d1d9 !important; }

/* Telemetry/Status box */
#status-box {
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.85em;
    color: #58a6ff; /* Blue accent for logs */
    background: #0d1117;
    padding: 10px;
    border-radius: 6px;
    border: 1px solid #30363d;
    margin-top: 10px;
}

/* Human Review panel */
#review-box textarea {
    font-family: 'JetBrains Mono', monospace;
    background-color: #0d1117 !important;
    border: 1px solid #d29922 !important; /* Yellow border to indicate action needed */
}

/* Header styling */
.header-title { font-weight: 800; letter-spacing: -0.5px; }
"""

with gr.Blocks(title="MEDIC ASTRO", theme=custom_theme, css=CSS) as demo:

    gr.Markdown(
        """
        <div style="text-align: center; margin-bottom: 20px;">
            <h1 class="header-title" style="color: #e6edf3;">🩺 MEDIC ASTRO</h1>
            <p style="color: #8b949e; font-size: 1.1em;">
                Secure Hospital Agent <span style="color: #58a6ff;">|</span>
                Policy <span style="color: #58a6ff;">|</span>
                HR <span style="color: #58a6ff;">|</span>
                Patient <span style="color: #58a6ff;">|</span>
                External Research
            </p>
        </div>
        """
    )

    thread_state = gr.State(value=None)

    with gr.Row():
        # ── Left: Chat & Audio ───────────────────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Agent Telemetry",
                height=550,
                buttons=["copy"],
            )

            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Voice Command (Whisper)",
                buttons=[]
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder="Type a secure query…",
                    label="Command Line",
                    scale=5,
                    lines=1,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                reset_btn = gr.Button("🔄 Clear Session", variant="secondary")

            status_box = gr.Markdown("_System Ready_", elem_id="status-box")

        # ── Right: Identity & HITL ───────────────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown("### 🔐 Access Control")
            user_id_box = gr.Textbox(
                label="User ID",
                value="EMP-000005",
                info="Padded 6-digit IDs (e.g. EMP-000001 CMO)",
            )
            clearance_box = gr.Dropdown(
                label="Clearance Level",
                choices=["Executive", "Department Head", "Physician", "Intern"],
                value="Intern",
            )
            department_box = gr.Dropdown(
                label="Department",
                choices=["HR", "Nursing", "Cardiology", "Radiology", "Oncology", "Administration"],
                value="Administration",
            )

            gr.Markdown("---")
            gr.Markdown("### 🧑‍⚕️ Human-in-the-Loop")
            review_box = gr.Textbox(
                label="Pending Action",
                lines=10,
                interactive=True,
                elem_id="review-box",
                placeholder="Awaiting interrupt... Draft will appear here.",
            )
            with gr.Row():
                approve_btn = gr.Button("✅ Approve", variant="primary")
                reject_btn  = gr.Button("❌ Reject",  variant="stop")

    # ── Wiring ────────────────────────────────────────────────────────
    send_btn.click(
        respond,
        inputs=[msg_box, chatbot, user_id_box, clearance_box, department_box, thread_state],
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )
    msg_box.submit(
        respond,
        inputs=[msg_box, chatbot, user_id_box, clearance_box, department_box, thread_state],
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )

    audio_input.stop_recording(
        transcribe_audio,
        inputs=[audio_input, chatbot, user_id_box, clearance_box, department_box, thread_state],
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )

    approve_btn.click(
        approve_review,
        inputs=[review_box, thread_state, chatbot],
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )
    reject_btn.click(
        reject_review,
        inputs=[review_box, thread_state, chatbot],
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )

    reset_btn.click(
        reset_session,
        outputs=[chatbot, msg_box, thread_state, status_box, review_box],
    )

print("✅ UI Rebuilt with Dark Theme.")
demo.launch(share=True, debug=False)